# EVO-ТУРНИР КОНФИГОВ LeanCore · Kaggle GPU T4×2 / P100

**Что делает**: ЭВОЛЮЦИОННЫЙ поиск (не перебор) по непрерывным скалярам (`mulr, ssfull, ssk,
ssalpha, lr`) — популяция развивается: селекция → кроссовер → мутации с адаптивным шагом σ
(правило 1/5), зал славы (HOF) по всей истории поколений. Методология проекта AIra/LeanCore:

- **CTRL** = рецепт чемпиона — вечный якорный элит в каждом поколении; фитнес = ΔPPL к CTRL
  ЭТОГО ЖЕ поколения (снимает дрейф между поколениями)
- Парность: битово общий инит (crc32-имена; torch с v3 инициализируется ТЕМ ЖЕ numpy-потоком —
  иначе гейт меряет шум инита, измерено: relΔ@40=2.0044% при пороге 2%), `negrng`-разделение
- Предохранители: авто-DQ «мёртвых» (`wn<3` после шага 100), NaN; прогнозы E1–E5 зафиксированы ДО запуска
- Стадии: **A qual** 16 конфигов @60 (отсев дна) → **B evo** популяция 8 × 3 поколения @200 ×2 сида
  → **C final** HOF-3 + CTRL @1500 × 4 сида
- Всё сохраняется в `evo_state.json` после каждой джобы → переживает таймаут сессии

## Как запустить
1. **Accelerator**: Settings → GPU T4 x2 (рекомендуется) или P100. Квота ~30 GPU-часов/нед (T4×2 тратит её быстрее — следите в интерфейсе).
2. **Данные**: три варианта (ячейка 3 разберёт сама): (а) приатачить Kaggle-датасет с `train.npy/val.npy/meta.json` корпуса `prep`; (б) положить их в `/kaggle/working/prep_data/`; (в) Internet ON → скачает с raw.githubusercontent нашей ветки.
3. **Run All**. Сначала пройдёт GPU-гейт честности (torch-GPU против numpy-CPU на 40 шагах CTRL), потом брекет.
4. По таймауту/завершении: **Save Version**. Для продолжения — новая сессия: Add Data → «Notebook Output Files» предыдущей версии → Run (state подхватится сам).

## Что вернуть в проект
Файлы из Output: `airaw/evo_work/EVO_SUMMARY.md`, `PREDICTIONS.md`, `evo_state.json`, и `results/*.jsonl` (+ `ckpt_*` финалистов). Сводку вставить в чат — я разберу против PREDICTIONS и запишу в TRICKS.


## 0 · Окружение и пути

In [ ]:
import os, sys, json, glob, time, shutil, subprocess, pathlib
ON_KAGGLE = os.path.exists("/kaggle/working")
BASE = "/kaggle/working/airaw" if ON_KAGGLE else os.path.abspath("./airaw_local")
os.makedirs(BASE, exist_ok=True)
print("kaggle:", ON_KAGGLE, "| BASE:", BASE, "| cpu:", os.cpu_count())
WORK = os.path.join(BASE, "evo_work")
os.makedirs(WORK, exist_ok=True)


## 1 · Файлы движка (lcxp / nano_lc_kg / train_torch / evo)

In [ ]:
open(f'{BASE}/lcxp.py','w').write("\"\"\"lcxp \u2014 backend-\u0448\u0438\u043c \u0434\u043b\u044f nano_lc: numpy (CPU) | cupy (GPU, Kaggle).\n\u0423\u043f\u0440\u0430\u0432\u043b\u0435\u043d\u0438\u0435: env LC_BACKEND = numpy | cupy  (\u043f\u043e \u0443\u043c\u043e\u043b\u0447\u0430\u043d\u0438\u044e numpy).\n\u041f\u0440\u0438\u043d\u0446\u0438\u043f: \u0442\u0435\u043d\u0437\u043e\u0440\u043d\u0430\u044f \u043c\u0430\u0442\u0435\u043c\u0430\u0442\u0438\u043a\u0430 \u2192 xp; \u0434\u0430\u043d\u043d\u044b\u0435/\u0441\u0435\u043c\u043f\u043b\u0438\u0440\u043e\u0432\u0430\u043d\u0438\u0435/\u0438\u043d\u0434\u0435\u043a\u0441\u044b-\u043a\u0430\u043d\u0434\u0438\u0434\u0430\u0442\u044b \u2192 hnp (host numpy).\n\u042d\u0442\u043e \u0433\u0430\u0440\u0430\u043d\u0442\u0438\u0440\u0443\u0435\u0442 \u041e\u0414\u0418\u041d\u0410\u041a\u041e\u0412\u042b\u0419 \u043f\u043e\u0440\u044f\u0434\u043e\u043a \u0434\u0430\u043d\u043d\u044b\u0445 \u0438 \u043d\u0435\u0433\u0430\u0442\u0438\u0432\u043e\u0432 \u043d\u0430 CPU \u0438 GPU (\u043f\u0430\u0440\u043d\u043e\u0441\u0442\u044c \u0431\u0440\u0435\u043a\u0435\u0442\u0430).\n\"\"\"\nimport os\nimport numpy as hnp\n\nBACKEND = os.environ.get(\"LC_BACKEND\", \"numpy\").strip().lower()\nif BACKEND == \"cupy\":\n    try:\n        import cupy as xp\n        import cupyx\n        on_gpu = True\n    except Exception as e:                                # noqa\n        print(f\"[lcxp] cupy \u043d\u0435\u0434\u043e\u0441\u0442\u0443\u043f\u0435\u043d ({e}); \u0444\u043e\u043b\u0431\u044d\u043a \u043d\u0430 numpy\", flush=True)\n        BACKEND = \"numpy\"\n        import numpy as xp\n        cupyx = None\n        on_gpu = False\nelse:\n    import numpy as xp\n    cupyx = None\n    on_gpu = False\n    BACKEND = \"numpy\"\n\nf32 = xp.float32\n\n\ndef to_dev(a):\n    return xp.asarray(a)\n\n\ndef asnumpy(a):\n    if on_gpu:\n        return xp.asnumpy(a)\n    return hnp.asarray(a)\n\n\ndef scatter_rows(dst, ids, src):\n    \"\"\"dst[ids] += src \u0441 \u0414\u0423\u0411\u041b\u0418\u041a\u0410\u0422\u0410\u041c\u0418 \u0432 ids.\n    numpy-\u0432\u0435\u0442\u043a\u0430 \u2014 \u0431\u0438\u0442\u0443\u0430\u043b\u044c\u043d\u043e \u0442\u0430 \u0436\u0435 reduceat-\u0441\u0445\u0435\u043c\u0430, \u0447\u0442\u043e \u0432 \u043e\u0440\u0438\u0433\u0438\u043d\u0430\u043b\u0435 (\u043f\u0430\u0440\u0438\u0442\u0435\u0442 \u0444\u043e\u0440\u043a\u0430).\n    cupy-\u0432\u0435\u0442\u043a\u0430 \u2014 \u0430\u0442\u043e\u043c\u0430\u0440\u043d\u044b\u0439 scatter_add (\u0440\u0435\u0437\u0443\u043b\u044c\u0442\u0430\u0442 \u044d\u043a\u0432\u0438\u0432\u0430\u043b\u0435\u043d\u0442\u0435\u043d \u043f\u043e \u0441\u0443\u043c\u043c\u0435, \u043f\u043e\u0440\u044f\u0434\u043e\u043a \u0441\u043b\u043e\u0436\u0435\u043d\u0438\u044f \u0438\u043d\u043e\u0439).\"\"\"\n    if on_gpu:\n        src2 = src.reshape(ids.shape[0], -1)\n        dst2 = dst.reshape(-1, src2.shape[-1])\n        cupyx.scatter_add(dst2, ids, src2)\n        return\n    src2 = src.reshape(ids.shape[0], -1)\n    order = hnp.argsort(ids, kind=\"stable\")\n    sids = ids[order]\n    bounds = hnp.flatnonzero(hnp.r_[True, sids[1:] != sids[:-1]])\n    dst[sids[bounds]] += hnp.add.reduceat(src2[order], bounds)\n\n\ndef save_npz(path, d):\n    \"\"\"\u0425\u043e\u0441\u0442-\u0441\u043e\u0445\u0440\u0430\u043d\u0435\u043d\u0438\u0435 \u0447\u0435\u043a\u043f\u043e\u0438\u043d\u0442\u0430 (\u0441 device-\u043a\u043e\u043d\u0432\u0435\u0440\u0441\u0438\u0435\u0439).\"\"\"\n    hnp.savez(path, **{k: asnumpy(v) for k, v in d.items()})\n\n\nTRUNK_EXCLUDE = (\"E\", \"U\", \"pos\")\n\n\ndef trunk_keys(d):\n    return [k for k, v in d.items() if v.ndim == 2 and k not in TRUNK_EXCLUDE and min(v.shape) >= 16]\n\n\ndef init_norms(d):\n    \"\"\"\u041d\u043e\u0440\u043c\u044b \u0442\u0440\u0430\u043d\u043a\u0430 \u043f\u043e\u0441\u043b\u0435 \u0438\u043d\u0438\u0446\u0438\u0430\u043b\u0438\u0437\u0430\u0446\u0438\u0438 (host-\u043a\u043e\u043f\u0438\u0438, \u0434\u0451\u0448\u0435\u0432\u043e).\"\"\"\n    ks = trunk_keys(d)\n    tot = 0.0\n    for k in ks:\n        v = asnumpy(d[k]).astype(hnp.float64)\n        tot += float((v * v).sum())\n    return tot ** 0.5\n\n\ndef trunk_ratio(d, n0):\n    \"\"\"\u2016W_trunk\u2016 / \u2016W_trunk_init\u2016: \u0430\u0432\u0442\u043e-\u0434\u0438\u0441\u043a\u0432\u0430\u043b\u0438\u0444\u0438\u043a\u0430\u0446\u0438\u044f \u00ab\u043c\u0451\u0440\u0442\u0432\u044b\u0445\u00bb \u043f\u0440\u043e\u0433\u043e\u043d\u043e\u0432 (<3 \u043a \u0448\u0430\u0433\u0443 100).\"\"\"\n    return init_norms(d) / max(n0, 1e-12)\n")
print('lcxp.py ok')

In [ ]:
open(f'{BASE}/nano_lc_kg.py','w').write("#!/usr/bin/env python3\n\"\"\"nano_lc \u2014 \u0441\u043e\u0431\u0441\u0442\u0432\u0435\u043d\u043d\u044b\u0439 \u043c\u0438\u043d\u0438\u043c\u0430\u043b\u044c\u043d\u044b\u0439 DL-\u0444\u0440\u0435\u0439\u043c\u0432\u043e\u0440\u043a \u043d\u0430 \u0447\u0438\u0441\u0442\u043e\u043c NumPy (\u0431\u0435\u0437 torch/TF).\n\u0412\u044b\u0432\u043e\u0434\u044b backward \u2014 \u0440\u0443\u0447\u043d\u044b\u0435 (\u0441\u043c. MATH.md), \u0432\u0441\u0435 \u0432\u0435\u0440\u0438\u0444\u0438\u0446\u0438\u0440\u043e\u0432\u0430\u043d\u044b FD-\u043f\u0440\u043e\u0432\u0435\u0440\u043a\u043e\u0439 gradcheck.py.\n\n\u041c\u043e\u0434\u0435\u043b\u0438: kind = attn | ema | hybrid (\u0431\u043b\u043e\u043a0 attn, \u043e\u0441\u0442\u0430\u043b\u044c\u043d\u044b\u0435 ema).\nadr_kf \u2208 (0,1]: \u0431\u043b\u043e\u043a\u0438 1..L-1 \u043c\u0430\u0440\u0448\u0440\u0443\u0442\u0438\u0437\u0438\u0440\u0443\u044e\u0442 top-k=round(adr_kf\u00b7T) \u0442\u043e\u043a\u0435\u043d\u043e\u0432 \u0447\u0435\u0440\u0435\u0437 \u043f\u043b\u0435\u0447\u043e\nmixer+FFN \u0441 \u0433\u0435\u0439\u0442\u043e\u043c \u03c3(s_t) (ADR, mixture-of-depths \u0441\u0442\u0438\u043b\u044c); k \u0434\u0435\u043b\u0438\u0442\u0441\u044f \u043d\u0430 T \u043e\u043a\u043d\u0430.\n\nCLI: python3 nano_lc.py --kind ema --tag run --steps 500 [--adr 0.55] [--initckpt x.npz]\n\"\"\"\nimport os, sys, json, math, time, argparse, zlib\nimport numpy as hnp\nfrom lcxp import xp, to_dev, asnumpy, on_gpu, save_npz, init_norms, trunk_ratio, scatter_rows\nnp = xp  # backend (numpy \u043d\u0430 CPU, cupy \u043d\u0430 GPU); host-\u043e\u043f\u0435\u0440\u0430\u0446\u0438\u0438 \u2014 \u0447\u0435\u0440\u0435\u0437 hnp\n\nPROF = {}      # in-situ \u043f\u0440\u043e\u0444\u0430\u0439\u043b\u0435\u0440: NANOLC_PROF=1 \u2192 \u043c\u0435\u0434\u0438\u0430\u043d\u044b \u043f\u043e \u0441\u0435\u043a\u0446\u0438\u044f\u043c \u0432 \u043a\u043e\u043d\u0446\u0435\ndef _tick(sec, t0):\n    PROF.setdefault(sec, []).append(time.process_time_ns() - t0)\n    return time.process_time_ns()\n\nROOT = os.path.dirname(os.path.abspath(__file__))\nf32 = np.float32\n\ndef scatter_add_rows(dst, ids, src):\n    \"\"\"\u0411\u044d\u043a\u0435\u043d\u0434-\u0434\u0435\u043b\u0435\u0433\u0430\u0442 (numpy: \u0441\u043e\u0440\u0442+reduceat \u043a\u0430\u043a \u0432 \u043e\u0440\u0438\u0433\u0438\u043d\u0430\u043b\u0435; cupy: \u0430\u0442\u043e\u043c\u0430\u0440\u043d\u044b\u0439 scatter_add).\"\"\"\n    scatter_rows(dst, ids.reshape(-1), src)\n\nclass Params:\n    def __init__(self): self.d, self.g = {}, {}\n    def add(self, name, shape, std=0.02, init=None):\n        rng = hnp.random.default_rng(zlib.crc32(name.encode()) & 0xFFFFFFFF)  # \u0434\u0435\u0442\u0435\u0440\u043c\u0438\u043d\u0438\u0437\u043c: hash() \u0441\u043e\u043b\u0438\u0442\u0441\u044f \u0437\u0430 \u043f\u0440\u043e\u0446\u0435\u0441\u0441\n        self.d[name] = to_dev(rng.normal(0, std, shape).astype(f32) if init is None else init(shape))\n        self.g[name] = np.zeros_like(self.d[name])\n    def zero(self):\n        for k in self.g: self.g[k][...] = 0\n\n# ---------------------------------------------------------------- core ops (fwd + hand-derived bwd)\n\n# ---- ctypes-\u044f\u0434\u0440\u0430 (lc_kernels.so): gelu / layernorm / softmax+CE \u2014 \u0441 numpy-\u0444\u043e\u043b\u0431\u044d\u043a\u043e\u043c ----\nimport ctypes as _ct\n_KS = None\ntry:\n    if on_gpu: raise OSError('gpu backend: ctypes-\u044f\u0434\u0440\u0430 \u0432\u044b\u043a\u043b\u044e\u0447\u0435\u043d\u044b')\n    _KS = _ct.CDLL(os.path.join(ROOT, \"lc_kernels.so\"))\n    _F = _ct.POINTER(_ct.c_float); _I = _ct.POINTER(_ct.c_int64)\n    _KS.k_gelu_fwd.argtypes = [_F, _F, _F, _ct.c_int64]\n    _KS.k_gelu_bwd.argtypes = [_F, _F, _F, _F, _ct.c_int64]\n    _KS.k_ln_fwd.argtypes = [_F, _F, _F, _F, _F, _F, _F, _ct.c_int64, _ct.c_int, _ct.c_float]\n    _KS.k_ln_bwd.argtypes = [_F, _F, _F, _F, _F, _F, _F, _ct.c_int64, _ct.c_int]\n    _KS.k_sce.argtypes = [_F, _I, _ct.c_int64, _ct.c_int64, _ct.c_float]\n    _KS.k_sce.restype = _ct.c_double\n    _KS.k_ema_fwd.argtypes = [_F, _F, _F, _F, _F, _ct.c_int, _ct.c_int, _ct.c_int]\n    _KS.k_ema_bwd.argtypes = [_F, _F, _F, _F, _F, _F, _F, _F, _ct.c_int, _ct.c_int, _ct.c_int]\nexcept OSError:\n    _KS = None\n\n\ndef _fp(a): return a.ctypes.data_as(_ct.POINTER(_ct.c_float))\n\ndef gelu(x):                                   # tanh-approx gelu: ctypes-\u044f\u0434\u0440\u043e \u0438\u043b\u0438 numpy\n    if _KS is not None and x.dtype == np.float32:\n        xc = np.ascontiguousarray(x, np.float32); n = xc.size\n        y = np.empty_like(xc); t = np.empty_like(xc)\n        _KS.k_gelu_fwd(_fp(xc), _fp(y), _fp(t), n)\n        return y, (xc, t)\n    c1 = np.float32(0.044715); c0 = np.float32(0.7978845608028654)\n    x2 = x * x\n    u = x2 * x; u *= c1; u += x; u *= c0\n    t = np.tanh(u)\n    y = t.copy(); y += np.float32(1.0); y *= x; y *= np.float32(0.5)\n    return y, (x, t)\ndef gelu_bwd(c, dy):                           # dx = dy\u00b7(\u00bd(1+t) + \u00bdx(1\u2212t\u00b2)\u00b7k0(1+3c\u2081x\u00b2))\n    x, t = c\n    if _KS is not None and dy.dtype == np.float32:\n        dyc = np.ascontiguousarray(dy, np.float32); n = x.size\n        dx = np.empty_like(x)\n        _KS.k_gelu_bwd(_fp(x), _fp(t), _fp(dyc), _fp(dx), n)\n        return dx\n    q = t.copy(); q *= t; q *= np.float32(-1.0); q += np.float32(1.0)   # 1\u2212t\u00b2\n    x2 = x * x; x2 *= np.float32(3.0 * 0.044715); x2 += np.float32(1.0) # 1+3c\u2081x\u00b2\n    q *= x2; q *= np.float32(0.5 * 0.7978845608028654); q *= x          # \u00bdk0\u00b7x\u00b7(1\u2212t\u00b2)(1+3c\u2081x\u00b2)\n    r = t.copy(); r += np.float32(1.0); r *= np.float32(0.5); r += q\n    r *= dy\n    return r\n\ndef layernorm(x, g, b, eps=1e-5):              # x: (...,N)\n    N = x.shape[-1]\n    if _KS is not None and x.dtype == np.float32:\n        xc = np.ascontiguousarray(x, np.float32)\n        R = xc.size // N\n        y = np.empty_like(xc); xhat = np.empty_like(xc)\n        mu = np.empty(R, np.float32); istd = np.empty(R, np.float32)\n        _KS.k_ln_fwd(_fp(xc), _fp(g), _fp(b), _fp(y), _fp(xhat), _fp(mu), _fp(istd), R, N, np.float32(eps))\n        return y, (xhat, istd.reshape(x.shape[:-1] + (1,)), g, N)\n    mu = x.mean(-1, keepdims=True); xc = x - mu\n    var = (xc**2).mean(-1, keepdims=True); istd = 1.0 / np.sqrt(var + eps)\n    xhat = xc * istd\n    return g * xhat + b, (xhat, istd, g, N)\ndef layernorm_bwd(c, dy):\n    xhat, istd, g, N = c\n    if _KS is not None and dy.dtype == np.float32:\n        dyc = np.ascontiguousarray(dy, np.float32)\n        R = dyc.size // N\n        dx = np.empty_like(dyc); dg = np.empty(N, np.float32); db = np.empty(N, np.float32)\n        _KS.k_ln_bwd(_fp(np.ascontiguousarray(g, np.float32)), _fp(dyc),\n                     _fp(np.ascontiguousarray(xhat, np.float32)),\n                     _fp(np.ascontiguousarray(istd.reshape(-1), np.float32)),\n                     _fp(dx), _fp(dg), _fp(db), R, N)\n        return dx, dg, db\n    dg = (dy * xhat).sum(axis=tuple(range(dy.ndim - 1)))\n    db = dy.sum(axis=tuple(range(dy.ndim - 1)))\n    v = dy * g                                                   # \u03b3 \u0414\u041e \u0440\u0435\u0434\u0443\u043a\u0446\u0438\u0439\n    dx = (istd / N) * (N * v - v.sum(-1, keepdims=True) - xhat * (v * xhat).sum(-1, keepdims=True))\n    return dx, dg, db\n\n# ---------------- sampled softmax (batch-candidates + bias-correction Q_c = 1\u2212(1\u2212q)^K) -------------\n# \u0413\u043e\u043b\u043e\u0432\u0430 H@E[cand].T \u043f\u043e m \u226a V \u043a\u0430\u043d\u0434\u0438\u0434\u0430\u0442\u0430\u043c; \u043b\u043e\u0433\u0438\u0442\u044b \u043a\u043e\u0440\u0440\u0435\u043a\u0442\u0438\u0440\u0443\u044e\u0442\u0441\u044f \u2212logQ_c (RAW/Jean-style):\n# E[\u2202L'/\u2202z] \u2248 \u2202L/\u2202z \u043f\u0440\u0438 \u043e\u0431\u0449\u0435\u043c \u043d\u0430\u0431\u043e\u0440\u0435 \u043a\u0430\u043d\u0434\u0438\u0434\u0430\u0442\u043e\u0432 \u043d\u0430 \u0431\u0430\u0442\u0447. \u0412\u0430\u043b \u2014 \u0432\u0441\u0435\u0433\u0434\u0430 \u043f\u043e\u043b\u043d\u044b\u0439 softmax.\n\ndef sampled_ce(H, y, E, cand, logcor, Ec=None):\n    \"\"\"H (B,T,D), y (B,T), cand (m,) sorted unique \u2287 targets, logcor (m,) = logQ.\n    \u2192 (loss, (dz, cand, Ec)); dz \u2014 \u0433\u0440\u0430\u0434\u0438\u0435\u043d\u0442 \u043f\u043e \u0441\u043a\u043e\u0440\u0440\u0435\u043a\u0442\u0438\u0440\u043e\u0432\u0430\u043d\u043d\u044b\u043c \u043b\u043e\u0433\u0438\u0442\u0430\u043c (p\u2212onehot)/(B\u00b7T).\n    Ec \u043c\u043e\u0436\u043d\u043e \u043f\u0435\u0440\u0435\u0434\u0430\u0442\u044c \u043f\u043e\u0441\u0447\u0438\u0442\u0430\u043d\u043d\u044b\u043c (\u043d\u0438\u0437\u043a\u043e\u0440\u0430\u043d\u0433\u043e\u0432\u044b\u0439 \u0441\u043b\u0443\u0447\u0430\u0439: U[cand]@P).\"\"\"\n    if Ec is None:\n        Ec = E[cand]                                 # (m,D)\n    lg = H @ Ec.T                                    # (B,T,m)\n    lg = lg - logcor[None, None, :]\n    lbl = np.searchsorted(cand, y)\n    loss, dz = softmax_ce(lg, lbl, destroy=True)\n    return loss, (dz, cand, Ec)\n\ndef sampled_ce_bwd(c, H):\n    dz, cand, Ec = c\n    dH = dz @ Ec                                     # (B,T,D)\n    dEc = dz.reshape(-1, dz.shape[-1]).T @ H.reshape(-1, H.shape[-1])   # (m,D)\n    return dH, dEc\n\nACTQ8 = False   # QAT-\u0440\u0435\u0436\u0438\u043c: \u043a\u0432\u0430\u043d\u0442\u043e\u0432\u0430\u043d\u0438\u0435 \u0430\u043a\u0442\u0438\u0432\u0430\u0446\u0438\u0439 int8 (\u0434\u0438\u043d\u0430\u043c\u0438\u0447\u0435\u0441\u043a\u0438\u0439 absmax \u043d\u0430 \u0441\u0442\u0440\u043e\u043a\u0443) \u043f\u0435\u0440\u0435\u0434 \u043c\u0430\u0442\u043c\u0443\u043b\u043e\u043c\n\ndef _actq(x):\n    am = np.abs(x).max(-1, keepdims=True)\n    am = np.where(am < 1e-8, np.ones_like(am), am)\n    return np.rint(x / am * 127) * (am / 127)\n\ndef linear(x, W):                              # (..., in) @ (in,out)\n    xq = _actq(x) if ACTQ8 else x\n    return xq @ W, (xq, W)                     # \u0433\u0440\u0430\u0434\u0438\u0435\u043d\u0442 \u043f\u043e \u043a\u0432\u0430\u043d\u0442\u043e\u0432\u0430\u043d\u043d\u043e\u043c\u0443 \u0432\u0445\u043e\u0434\u0443 (STE-\u0441\u0442\u0430\u043d\u0434\u0430\u0440\u0442)\ndef linear_bwd(c, dy):\n    x, W = c\n    dw = x.reshape(-1, x.shape[-1]).T @ dy.reshape(-1, dy.shape[-1])\n    return dy @ W.T, dw\n\n\ndef kron_pair_facs(in_d, out_d):\n    \"\"\"\u0424\u0430\u043a\u0442\u043e\u0440\u044b in=n1\u00b7n2 (n1>=n2), out=m1\u00b7m2 (m1<=m2), \u0441\u0442\u043e\u0438\u043c\u043e\u0441\u0442\u044c m1\u00b7n2\u00b7(n1+m2) \u043c\u0438\u043d\u0438\u043c\u0430\u043b\u044c\u043d\u0430.\"\"\"\n    import math as _m\n    def facs(n, want_min_first):\n        best, bd = (n, 1), 1e9\n        for a in range(1, int(_m.sqrt(n)) + 1):\n            if n % a == 0:\n                b = n // a\n                d = (a - b) if want_min_first else (b - a)\n                if abs(a - b) < bd:\n                    bd = abs(a - b); best = (a, b) if want_min_first else (b, a)\n        return best\n    n2, n1 = facs(in_d, False)     # n1 \u2014 \u0431\u043e\u043b\u044c\u0448\u0438\u0439 \u0444\u0430\u043a\u0442\u043e\u0440 \u0432\u0445\u043e\u0434\u0430\n    m1, m2 = facs(out_d, True)     # m1 \u2014 \u043c\u0435\u043d\u044c\u0448\u0438\u0439 \u0444\u0430\u043a\u0442\u043e\u0440 \u0432\u044b\u0445\u043e\u0434\u0430\n    return n1, n2, m1, m2\n\ndef kron_fwd(x2d, As, Bs):\n    \"\"\"x2d (P, n1\u00b7n2) \u2192 (y (P, m1\u00b7m2), cache). Y = \u03a3_k A_k X B_k\u1d40, vec row-major \u0441 \u043e\u0431\u0435\u0438\u0445 \u0441\u0442\u043e\u0440\u043e\u043d.\"\"\"\n    P = x2d.shape[0]\n    n1, n2 = As[0].shape[1], Bs[0].shape[1]\n    m1, m2 = As[0].shape[0], Bs[0].shape[0]\n    X = x2d.reshape(P, n1, n2)\n    Y = np.zeros((P, m1, m2), x2d.dtype)\n    for A, B in zip(As, Bs):\n        t = X.transpose(0, 2, 1).reshape(P * n2, n1) @ A.T\n        t = t.reshape(P, n2, m1).transpose(0, 2, 1).reshape(P * m1, n2) @ B.T\n        Y += t.reshape(P, m1, m2)\n    return Y.reshape(P, m1 * m2), (X, As, Bs)\n\ndef kron_bwd(c, dy2d):\n    \"\"\"dy2d (P, m1\u00b7m2) \u2192 (dX (P, n1\u00b7n2), [dA_k], [dB_k]).\n    dA_k = \u03a3_p G B X\u1d40;  dB_k = \u03a3_p G\u1d40 A X;  dX = \u03a3_k A\u1d40 G B.\"\"\"\n    X, As, Bs = c\n    P = X.shape[0]\n    m1, m2 = As[0].shape[0], Bs[0].shape[0]\n    G = dy2d.reshape(P, m1, m2)\n    Gt = G.transpose(0, 2, 1)\n    dX = np.zeros_like(X)\n    dAs, dBs = [], []\n    for A, B in zip(As, Bs):\n        dAs.append(np.einsum('pmr,pnr->mn', G @ B, X))\n        dBs.append(np.einsum('pmn,pnq->mq', Gt @ A, X))\n        dX += A.T @ G @ B\n    return dX.reshape(P, X.shape[1] * X.shape[2]), dAs, dBs\n\ndef softmax_ce(logits, y, destroy=False):      # logits (B,T,V), y (B,T) -> (loss, dlogits); destroy=True \u2192 \u043f\u0438\u0448\u0435\u043c dz \u043f\u043e\u0432\u0435\u0440\u0445 logits\n    B, T, V = logits.shape\n    if _KS is not None and logits.dtype == np.float32:\n        dz = logits if (destroy and logits.flags['C_CONTIGUOUS']) else np.ascontiguousarray(logits, np.float32).copy()\n        yy = np.ascontiguousarray(y, np.int64).reshape(-1)\n        nll = _KS.k_sce(_fp(dz), _ct.cast(yy.ctypes.data, _ct.POINTER(_ct.c_int64)),\n                        B * T, V, np.float32(1.0 / (B * T)))\n        return nll, dz\n    z = logits - logits.max(-1, keepdims=True)\n    e = np.exp(z); p = e / e.sum(-1, keepdims=True)\n    nll = -np.log(p.reshape(-1, V)[np.arange(B*T), y.reshape(-1)] + 1e-12).mean()\n    dz = p\n    dz.reshape(-1, V)[np.arange(B*T), y.reshape(-1)] -= 1.0\n    dz /= (B * T)\n    return nll, dz\n\ndef attention(X, Wqkv, Wo, h):                 # X (B,T,D), causal MHA\n    B, T, D = X.shape; dh = D // h\n    Z = X @ Wqkv                                # (B,T,3D)\n    Q = Z[..., :D].reshape(B, T, h, dh).transpose(0, 2, 1, 3)\n    K = Z[..., D:2*D].reshape(B, T, h, dh).transpose(0, 2, 1, 3)\n    V = Z[..., 2*D:].reshape(B, T, h, dh).transpose(0, 2, 1, 3)\n    S = (Q @ K.transpose(0, 1, 3, 2)) / math.sqrt(dh)\n    mask = np.triu(np.ones((T, T), bool), 1)\n    S = np.where(mask, -1e30, S)\n    A = np.exp(S - S.max(-1, keepdims=True)); A /= A.sum(-1, keepdims=True)\n    Y = A @ V                                   # (B,h,T,dh)\n    Yc = Y.transpose(0, 2, 1, 3).reshape(B, T, D)\n    out = Yc @ Wo\n    return out, (X, Wqkv, Wo, Q, K, V, A, Yc)\ndef attention_bwd(c, dY):\n    X, Wqkv, Wo, Q, K, V, A, Yc = c\n    B, h, T, dh = Q.shape; D = Wo.shape[1]\n    dWo = Yc.reshape(-1, D).T @ dY.reshape(-1, D)\n    dYc = dY @ Wo.T\n    dY2 = dYc.reshape(B, T, h, dh).transpose(0, 2, 1, 3)\n    dA = dY2 @ V.transpose(0, 1, 3, 2)\n    dV = A.transpose(0, 1, 3, 2) @ dY2\n    dS = A * (dA - (A * dA).sum(-1, keepdims=True)) / math.sqrt(dh)\n    dQ = dS @ K\n    dK = dS.transpose(0, 1, 3, 2) @ Q\n    dQ_ = dQ.transpose(0, 2, 1, 3).reshape(B, T, D)\n    dK_ = dK.transpose(0, 2, 1, 3).reshape(B, T, D)\n    dV_ = dV.transpose(0, 2, 1, 3).reshape(B, T, D)\n    dZ = np.concatenate([dQ_, dK_, dV_], axis=-1)\n    dWqkv = X.reshape(-1, D).T @ dZ.reshape(-1, 3 * D)\n    return dZ @ Wqkv.T, dWqkv, dWo\n\n_EMA_CACHE = {}                                   # (T,D,a) \u2192 M_forward, L_backward\n\ndef _ema_mats(a, T, D, dt):\n    key = (T, D, a.tobytes(), dt)\n    got = _EMA_CACHE.get(key)\n    if got is not None: return got\n    if len(_EMA_CACHE) > 12: _EMA_CACHE.clear()   # a \u043c\u0435\u043d\u044f\u0435\u0442\u0441\u044f \u043a\u0430\u0436\u0434\u044b\u0439 \u0448\u0430\u0433 \u2192 \u043d\u0435 \u043a\u043e\u043f\u0438\u043c\n    tt = np.arange(T)\n    dd = tt[:, None] - tt[None, :]                # t\u2212k\n    mf = dd >= 0\n    ad = np.abs(dd)[:, :, None].astype(dt)\n    Pf = a[None, None, :] ** ad\n    M = Pf * mf[:, :, None] * (1.0 - a)[None, None, :]         # h_t = \u03a3_k a^{t\u2212k}(1\u2212a)x_k, k\u2264t\n    Lb = Pf * (dd <= 0)[:, :, None]                             # lam_t = \u03a3_s a^{s\u2212t}\u00b7dH_s, s\u2265t\n    _EMA_CACHE[key] = (M, Lb)\n    return M, Lb\n\ndef ema_mix(X, th, sc):                        # h_t = a\u00b7h_{t-1} + (1-a)\u00b7x_t; y = sc\u2299h\n    a = (1.0 / (1.0 + np.exp(-th))).astype(X.dtype)\n    B, T, D = X.shape\n    if _KS is not None and X.dtype == np.float32:\n        Xc = np.ascontiguousarray(X)\n        H = np.empty_like(Xc); Y = np.empty_like(Xc)\n        ac = np.ascontiguousarray(a); scc = np.ascontiguousarray(sc)\n        _KS.k_ema_fwd(_fp(Xc), _fp(ac), _fp(scc), _fp(Y), _fp(H), B, T, D)\n        return Y, (Xc, H, a, sc)\n    M, _ = _ema_mats(a, T, D, X.dtype)\n    H = np.einsum('tkd,bkd->btd', M, X, optimize=True)\n    return H * sc, (X, H, a, sc)\n\ndef ema_mix_bwd(c, dY):\n    X, H, a, sc = c\n    B, T, D = X.shape\n    if _KS is not None and dY.dtype == np.float32:\n        dYc = np.ascontiguousarray(dY)\n        dX = np.empty_like(Xc_ := np.ascontiguousarray(X))\n        dth = np.empty(D, np.float32); dsc = np.empty(D, np.float32)\n        _KS.k_ema_bwd(_fp(Xc_), _fp(np.ascontiguousarray(H)), _fp(np.ascontiguousarray(a)),\n                      _fp(np.ascontiguousarray(sc)), _fp(dYc), _fp(dX), _fp(dth), _fp(dsc), B, T, D)\n        return dX, dth, dsc\n    dH = dY * sc\n    dsc = (dY * H).sum(axis=(0, 1))\n    M, Lb = _ema_mats(a, T, D, X.dtype)\n    lam = np.einsum('tsd,bsd->btd', Lb, dH, optimize=True)       # lam_t = \u03a3_{s\u2265t} a^{s-t} dH_s\n    dX = (1.0 - a)[None, None, :] * lam\n    hp = np.zeros_like(H); hp[:, 1:] = H[:, :-1]                 # h_{t-1}\n    da = (lam * (hp - X)).sum((0, 1))\n    dth = da * a * (1.0 - a)\n    return dX, dth, dsc\n\n# ---------------------------------------------------------------- model\nclass NanoGPT:\n    def __init__(self, V, D=192, L=4, h=6, ff=576, T=96, kind=\"attn\", adr_kf=None, moe_e=1, mtp_w=0.0, erank=0, kronfc=0):  # noqa\n        self.V, self.D, self.L, self.h, self.T, self.ff, self.moe_e, self.mtp_w = V, D, L, h, T, ff, moe_e, mtp_w\n        self.kind, self.adr_kf, self.erank, self.kronfc = kind, adr_kf, erank, kronfc\n        p = self.p = Params()\n        if erank > 0:                                  # \u043d\u0438\u0437\u043a\u043e\u0440\u0430\u043d\u0433\u043e\u0432\u044b\u0439 tied E = U(V,r) @ P(r,D)\n            sg = (0.02 ** 2 / erank) ** 0.25           # Var(E_ij) = r\u00b7\u03c3\u2074 = 0.02\u00b2 \u2192 \u0447\u0435\u0441\u0442\u043d\u044b\u0439 init\n            p.add(\"U\", (V, erank), std=sg); p.add(\"P\", (erank, D), std=sg)\n        else:\n            p.add(\"E\", (V, D))\n        p.add(\"pos\", (T, D))\n        self.blocks = []\n        for i in range(L):\n            kind_i = \"attn\" if (kind == \"attn\" or (kind == \"hybrid\" and i == 0)) else (\"delta\" if kind == \"delta\" else \"ema\")\n            b = {\"kind\": kind_i, \"routed\": bool(adr_kf) and i > 0}\n            b[\"kind\"] = kind_i\n            self.blocks.append(b)\n            p.add(f\"b{i}.ln1g\", (D,), init=lambda s: np.ones(s, f32))\n            p.add(f\"b{i}.ln1b\", (D,), init=lambda s: np.zeros(s, f32))\n            if kind_i == \"attn\":\n                p.add(f\"b{i}.Wqkv\", (D, 3 * D)); p.add(f\"b{i}.Wo\", (D, D))\n            else:\n                p.add(f\"b{i}.th\", (D,), init=lambda s: np.zeros(s, f32))     # a=\u03c3(0)=0.5\n                p.add(f\"b{i}.sc\", (D,), init=lambda s: np.ones(s, f32))\n                if kind_i == \"delta\":\n                    p.add(f\"b{i}.braw\", (), init=lambda s: np.array(-2.0, f32))   # \u03b2(0)=0.119 \u2014 \u043c\u044f\u0433\u043a\u0430\u044f \u043a\u043e\u0440\u0440\u0435\u043a\u0446\u0438\u044f\n                    p.add(f\"b{i}.Wg\", (D, D))                                      # KDA output gate\n                p.add(f\"b{i}.Wm\", (D, D))\n            p.add(f\"b{i}.ln2g\", (D,), init=lambda s: np.ones(s, f32))\n            p.add(f\"b{i}.ln2b\", (D,), init=lambda s: np.zeros(s, f32))\n            if self.moe_e > 1:\n                E, F2 = self.moe_e, ff // 2\n                p.add(f\"b{i}.wr\", (D, E))\n                p.add(f\"b{i}.be\", (E,), init=lambda s: np.zeros(s, f32))\n                p.add(f\"b{i}.w1\", (E, D, F2))\n                p.add(f\"b{i}.w2\", (E, F2, D))\n            else:\n                if self.kronfc > 0:\n                    n1, n2, m1, m2 = kron_pair_facs(D, ff)\n                    o1, o2, q1, q2 = kron_pair_facs(ff, D)\n                    def kinit(shape):\n                        return hnp.random.default_rng(1).normal(0, (2.0 / (shape[0]*shape[1]))**0.25, shape).astype(f32)\n                    for k_ in range(self.kronfc):\n                        p.add(f\"b{i}.fc1A{k_}\", (m1, n1)); p.add(f\"b{i}.fc1B{k_}\", (m2, n2))\n                        p.add(f\"b{i}.fc2A{k_}\", (q1, o1)); p.add(f\"b{i}.fc2B{k_}\", (q2, o2))\n                    self._kfacs = (n1, n2, m1, m2, o1, o2, q1, q2)\n                else:\n                    p.add(f\"b{i}.fc1\", (D, ff)); p.add(f\"b{i}.fc2\", (ff, D))\n            if b[\"routed\"]:\n                p.add(f\"b{i}.rw\", (D,))\n                p.add(f\"b{i}.rb\", (), init=lambda s: np.array(1.5, f32))\n        p.add(\"lnfg\", (D,), init=lambda s: np.ones(s, f32))\n        p.add(\"lnfb\", (D,), init=lambda s: np.zeros(s, f32))\n        if mtp_w > 0:\n            p.add(\"lnm_g\", (D,), init=lambda s: np.ones(s, f32))\n            p.add(\"lnm_b\", (D,), init=lambda s: np.zeros(s, f32))\n            p.add(\"Wmtp\", (D, D))\n\n    def _arm_forward(self, X2, i, b):\n        \"\"\"\u043f\u043b\u0435\u0447\u043e \u0431\u043b\u043e\u043a\u0430: ln1 \u2192 mixer(+Wm/attn) \u2192 +resid \u2192 ln2 \u2192 ffn \u2192 delta=mix+o2\"\"\"\n        p = self.p.d; c = {}\n        ln_out, c[\"ln1\"] = layernorm(X2, p[f\"b{i}.ln1g\"], p[f\"b{i}.ln1b\"])\n        if b[\"kind\"] == \"attn\":\n            mix, c[\"attn\"] = attention(ln_out, p[f\"b{i}.Wqkv\"], p[f\"b{i}.Wo\"], self.h)\n        elif b[\"kind\"] == \"delta\":\n            em, c[\"ema\"] = delta_mix(ln_out, p[f\"b{i}.th\"], p[f\"b{i}.sc\"], p[f\"b{i}.braw\"], p[f\"b{i}.Wg\"])\n            mix, c[\"wm\"] = linear(em, p[f\"b{i}.Wm\"])\n        else:\n            em, c[\"ema\"] = ema_mix(ln_out, p[f\"b{i}.th\"], p[f\"b{i}.sc\"])\n            mix, c[\"wm\"] = linear(em, p[f\"b{i}.Wm\"])\n        H2 = X2 + mix\n        ln_out2, c[\"ln2\"] = layernorm(H2, p[f\"b{i}.ln2g\"], p[f\"b{i}.ln2b\"])\n        c[\"fc1_in\"] = ln_out2\n        if self.moe_e > 1:\n            c[f\"moe_be_{i}\"] = p[f\"b{i}.be\"].copy()\n            o2, c[\"moe\"] = moe_ffn(ln_out2, p[f\"b{i}.wr\"], p[f\"b{i}.be\"], p[f\"b{i}.w1\"], p[f\"b{i}.w2\"], i)\n        elif self.kronfc > 0:\n            P_, B_, T_ = ln_out2.shape[0] * ln_out2.shape[1], ln_out2.shape[0], ln_out2.shape[1]\n            n1, n2, m1, m2, o1, o2, q1, q2 = self._kfacs\n            z1, c[\"kron1\"] = kron_fwd(ln_out2.reshape(P_, self.D),\n                                      [p[f\"b{i}.fc1A{k_}\"] for k_ in range(self.kronfc)],\n                                      [p[f\"b{i}.fc1B{k_}\"] for k_ in range(self.kronfc)])\n            g1, c[\"gelu\"] = gelu(z1.reshape(B_, T_, self.ff))\n            z2, c[\"kron2\"] = kron_fwd(g1.reshape(P_, self.ff),\n                                      [p[f\"b{i}.fc2A{k_}\"] for k_ in range(self.kronfc)],\n                                      [p[f\"b{i}.fc2B{k_}\"] for k_ in range(self.kronfc)])\n            o2 = z2.reshape(B_, T_, self.D)          # fc2-cache \u043d\u0435 \u043d\u0443\u0436\u0435\u043d \u2014 kron bwd \u0441\u0432\u043e\u0439\n        else:\n            g1, c[\"gelu\"] = gelu(ln_out2 @ p[f\"b{i}.fc1\"])\n            o2, c[\"fc2\"] = linear(g1, p[f\"b{i}.fc2\"])\n        c[\"delta\"] = mix + o2\n        return mix + o2, c\n\n    def _arm_backward(self, c, dD, i, b):\n        \"\"\"dD = dL/dDelta \u2192 \u0432\u043e\u0437\u0432\u0440\u0430\u0449\u0430\u0435\u0442 dL/dX2 (\u0432\u0445\u043e\u0434 \u043f\u043b\u0435\u0447\u0430).\"\"\"\n        p, g = self.p.d, self.p.g\n        if self.moe_e > 1:\n            dxm, dwr, dbe, dw1, dw2 = moe_ffn_bwd(c[\"moe\"], dD)\n            g[f\"b{i}.wr\"] += dwr; g[f\"b{i}.w1\"] += dw1; g[f\"b{i}.w2\"] += dw2\n            dln2 = dxm\n        elif self.kronfc > 0:\n            P_ = dD.shape[0] * dD.shape[1]\n            n1, n2, m1, m2, o1, o2, q1, q2 = self._kfacs\n            dX2f, dA2s, dB2s = kron_bwd(c[\"kron2\"], dD.reshape(P_, self.D))\n            for k_ in range(self.kronfc):\n                g[f\"b{i}.fc2A{k_}\"] += dA2s[k_]; g[f\"b{i}.fc2B{k_}\"] += dB2s[k_]\n            dpre = gelu_bwd(c[\"gelu\"], dX2f.reshape(dD.shape[0], dD.shape[1], self.ff))\n            dln24, dA1s, dB1s = kron_bwd(c[\"kron1\"], dpre.reshape(P_, self.ff))\n            for k_ in range(self.kronfc):\n                g[f\"b{i}.fc1A{k_}\"] += dA1s[k_]; g[f\"b{i}.fc1B{k_}\"] += dB1s[k_]\n            dln2 = dln24.reshape(dD.shape[0], dD.shape[1], self.D)\n        else:\n            dX2f, dW2 = linear_bwd(c[\"fc2\"], dD); g[f\"b{i}.fc2\"] += dW2\n            dpre = gelu_bwd(c[\"gelu\"], dX2f)\n            g[f\"b{i}.fc1\"] += c[\"fc1_in\"].reshape(-1, self.D).T @ dpre.reshape(-1, self.ff)\n            dln2 = dpre @ p[f\"b{i}.fc1\"].T\n        dx2, dg2, db2 = layernorm_bwd(c[\"ln2\"], dln2); g[f\"b{i}.ln2g\"] += dg2; g[f\"b{i}.ln2b\"] += db2\n        dmix = dD + dx2                            # mix \u043f\u0438\u0442\u0430\u0435\u0442 \u0394 \u0438 H2 (\u0432\u0445\u043e\u0434 ln2)\n        if b[\"kind\"] == \"attn\":\n            dxm, dWqkv, dWo = attention_bwd(c[\"attn\"], dmix)\n            g[f\"b{i}.Wqkv\"] += dWqkv; g[f\"b{i}.Wo\"] += dWo\n        elif b[\"kind\"] == \"delta\":\n            dxe, dWm = linear_bwd(c[\"wm\"], dmix); g[f\"b{i}.Wm\"] += dWm\n            dxm, dth, dsc, dbraw, dWg = delta_mix_bwd(c[\"ema\"], dxe)\n            g[f\"b{i}.th\"] += dth; g[f\"b{i}.sc\"] += dsc; g[f\"b{i}.braw\"] += dbraw; g[f\"b{i}.Wg\"] += dWg\n        else:\n            dxe, dWm = linear_bwd(c[\"wm\"], dmix); g[f\"b{i}.Wm\"] += dWm\n            dxm, dth, dsc = ema_mix_bwd(c[\"ema\"], dxe)\n            g[f\"b{i}.th\"] += dth; g[f\"b{i}.sc\"] += dsc\n        dx1, dg1, db1 = layernorm_bwd(c[\"ln1\"], dxm); g[f\"b{i}.ln1g\"] += dg1; g[f\"b{i}.ln1b\"] += db1\n        return dx2 + dx1\n\n    def forward(self, ids):\n        \"\"\"\u2192 (H \u043f\u043e\u0441\u043b\u0435 \u0444\u0438\u043d\u0430\u043b\u044c\u043d\u043e\u0433\u043e LN, caches)\"\"\"\n        p = self.p.d\n        B, T = ids.shape\n        if self.erank > 0:\n            H = p[\"U\"][ids] @ p[\"P\"] + p[\"pos\"][None, :T]\n        else:\n            H = p[\"E\"][ids] + p[\"pos\"][None, :T]\n        blocks_c = []\n        for i, b in enumerate(self.blocks):\n            bc = {}\n            if b[\"routed\"]:\n                Xin = H\n                s = Xin @ p[f\"b{i}.rw\"] + p[f\"b{i}.rb\"]                # (B,T)\n                k = max(1, int(round(self.adr_kf * T)))\n                idx = np.argsort(-s, axis=1)[:, :k]; idx.sort(axis=1)\n                bi = np.arange(B)[:, None]\n                xs = Xin[bi, idx]                                     # (B,k,D)\n                g = 1.0 / (1.0 + np.exp(-s[bi, idx]))                 # (B,k)\n                delta, arm_c = self._arm_forward(xs, i, b)\n                H = H.copy()\n                H[bi, idx] = H[bi, idx] + delta * g[..., None]\n                bc[\"route\"] = (Xin, s, idx, g, k, arm_c)\n            else:\n                bc[\"Xin\"] = H\n                delta, arm_c = self._arm_forward(H, i, b)\n                bc[\"arm\"] = arm_c\n                H = H + delta\n            blocks_c.append(bc)\n        out, c_lnf = layernorm(H, p[\"lnfg\"], p[\"lnfb\"])\n        return out, (ids, T, blocks_c, c_lnf)\n\n    def logits(self, H):\n        if self.erank > 0:\n            return (H @ self.p.d[\"P\"].T) @ self.p.d[\"U\"].T            # tied lowrank head\n        return H @ self.p.d[\"E\"].T                   # tied head\n\n    def backward(self, H, caches, dlogits=None, dH_aux=None, head_cache=None):\n        p, g = self.p.d, self.p.g\n        ids, T, blocks_c, c_lnf = caches\n        _t = time.process_time_ns()\n        if head_cache is not None:                      # sampled softmax: \u0440\u0430\u0437\u0440\u0435\u0436\u0435\u043d\u043d\u044b\u0439 \u043f\u0443\u0442\u044c\n            cand, dz, Ec = head_cache\n            if self.erank > 0:                          # Ec = U[cand]@P \u0431\u044b\u043b \u0444\u043e\u0440\u0432\u0430\u0440\u0434-\u043a\u043e\u043f\u0438\u0435\u0439; \u0433\u0440\u0430\u0434\u0438\u0435\u043d\u0442 \u043f\u043e U,P\n                PH = H @ p[\"P\"].T                       # (B,T,r)\n                Uc = p[\"U\"][cand]                       # (m,r)\n                g[\"U\"][cand] += dz.reshape(-1, dz.shape[-1]).T @ PH.reshape(-1, self.erank)\n                dPH = dz @ Uc                           # (B,T,r)\n                g[\"P\"] += dPH.reshape(-1, self.erank).T @ H.reshape(-1, self.D)\n                dH = dPH @ p[\"P\"]\n            else:\n                g[\"E\"][cand] += dz.reshape(-1, dz.shape[-1]).T @ H.reshape(-1, self.D)   # cand \u0443\u043d\u0438\u043a\u0430\u043b\u0435\u043d \u2192 fancy-add\n                dH = dz @ Ec\n        elif self.erank > 0:                            # \u043f\u043e\u043b\u043d\u0430\u044f \u0433\u043e\u043b\u043e\u0432\u0430, \u043d\u0438\u0437\u043a\u0438\u0439 \u0440\u0430\u043d\u0433\n            PH = H @ p[\"P\"].T\n            g[\"U\"] += dlogits.reshape(-1, self.V).T @ PH.reshape(-1, self.erank)\n            dPH = dlogits @ p[\"U\"]                      # (B,T,r)\n            g[\"P\"] += dPH.reshape(-1, self.erank).T @ H.reshape(-1, self.D)\n            dH = dPH @ p[\"P\"]\n        else:\n            g[\"E\"] += dlogits.reshape(-1, self.V).T @ H.reshape(-1, self.D)\n            dH = dlogits @ p[\"E\"]\n        _t = _tick(\"bwd_head\", _t)\n        if dH_aux is not None: dH = dH + dH_aux            # MTP: aux-\u0433\u0440\u0430\u0434\u0438\u0435\u043d\u0442 \u0443\u0440\u043e\u0432\u043d\u044f H (\u043f\u043e\u0441\u043b\u0435 lnf)\n        dH, dg, db = layernorm_bwd(c_lnf, dH); g[\"lnfg\"] += dg; g[\"lnfb\"] += db\n        _t = _tick(\"bwd_body\", _t)\n        for i in reversed(range(self.L)):\n            bc = blocks_c[i]; b = self.blocks[i]\n            if b[\"routed\"]:\n                Xin, s, idx, gsel, k, arm_c = bc[\"route\"]\n                B = Xin.shape[0]; bi = np.arange(B)[:, None]\n                dH_sel = dH[bi, idx]                                  # (B,k,D)\n                delta = arm_c[\"delta\"]\n                # dL/dg = <dH_sel, delta>; dL/ds = \u00b7g(1\u2212g)\n                dg_ = (dH_sel * delta).sum(-1)\n                ds_ = dg_ * gsel * (1 - gsel)\n                g[f\"b{i}.rw\"] += (ds_[..., None] * Xin[bi, idx]).sum((0, 1))\n                g[f\"b{i}.rb\"] += ds_.sum()\n                # dL/dx_sel = dH_sel + J_arm\u1d40(g\u2299dH_sel)   [dDelta = g\u2299dH]\n                dx_arm = self._arm_backward(arm_c, dH_sel * gsel[..., None], i, b)\n                dH[bi, idx] = dH_sel + dx_arm\n                dH[bi, idx] += ds_[..., None] * p[f\"b{i}.rw\"][None, None, :]   # \u043f\u0443\u0442\u044c \u0447\u0435\u0440\u0435\u0437 \u0441\u043a\u043e\u0440\n                continue\n            dX = self._arm_backward(bc[\"arm\"], dH, i, b)\n            dH = dH + dX\n        _t = _tick(\"bwd_blocks\", _t)\n        if self.erank > 0:                              # lookup-\u043f\u0443\u0442\u044c: x0 = U[ids]@P\n            Ui = p[\"U\"][ids].reshape(-1, self.erank)\n            dHf = dH.reshape(-1, self.D)\n            scatter_add_rows(g[\"U\"], ids, dHf @ p[\"P\"].T)\n            g[\"P\"] += Ui.T @ dHf\n        else:\n            scatter_add_rows(g[\"E\"], ids, dH)   # \u0433\u0440\u0430\u0434\u0438\u0435\u043d\u0442 \u044d\u043c\u0431\u0435\u0434\u0434\u0438\u043d\u0433\u0430 (lookup-\u043f\u0443\u0442\u044c, \u0432\u0435\u043a\u0442\u043e\u0440\u043d\u044b\u0439)\n        g[\"pos\"] += dH.sum(0)\n        _t = _tick(\"bwd_embscat\", _t)\n        return H\n\nclass AdamW:\n    def __init__(self, params, lr=6e-4, b1=0.9, b2=0.95, wd=0.1, eps=1e-8, wdmask=False):\n        self.p = params; self.b1, self.b2, self.wd, self.eps = b1, b2, wd, eps\n        # wdmask: decoupled-wd \u0442\u043e\u043b\u044c\u043a\u043e \u043d\u0430 \u0441\u043a\u0440\u044b\u0442\u044b\u0445 2D-\u043c\u0430\u0442\u0440\u0438\u0446\u0430\u0445; \u041d\u0415 \u043d\u0430 tied-\u044d\u043c\u0431\u0435\u0434\u0434\u0438\u043d\u0433\u0435 \"E\",\n        # LN-gains/\u0431\u0438\u0430\u0441\u0430\u0445/\u0441\u043a\u0430\u043b\u044f\u0440\u0430\u0445 (ndim<2). \u041a\u043b\u0430\u0441\u0441\u0438\u0447\u0435\u0441\u043a\u0430\u044f \u043f\u0440\u0430\u0432\u043a\u0430 AdamW \u2192 \u043e\u0431\u044b\u0447\u043d\u043e +\u0441\u0442\u0430\u0431\u0438\u043b\u044c\u043d\u043e\u0441\u0442\u044c.\n        self.wdmask = wdmask\n        self.m = {k: np.zeros_like(v) for k, v in params.d.items()}\n        self.v = {k: np.zeros_like(v) for k, v in params.d.items()}\n        self.t = 0\n    def step(self, lr):\n        self.t += 1\n        for k in self.p.d:\n            g = self.p.g[k]\n            self.m[k] = self.b1 * self.m[k] + (1 - self.b1) * g\n            self.v[k] = self.b2 * self.v[k] + (1 - self.b2) * g * g\n            mh = self.m[k] / (1 - self.b1 ** self.t)\n            vh = self.v[k] / (1 - self.b2 ** self.t)\n            wd = 0.0 if (self.wdmask and (self.p.d[k].ndim < 2 or k in (\"E\", \"U\", \"P\"))) else self.wd\n            self.p.d[k] -= lr * (mh / (np.sqrt(vh) + self.eps) + wd * self.p.d[k])\n        self.p.zero()\n\nclass MuonW:\n    \"\"\"Muon (Keller Jordan): momentum + Newton\u2013Schulz \u043e\u0440\u0442\u043e\u0433\u043e\u043d\u0430\u043b\u0438\u0437\u0430\u0446\u0438\u044f \u0434\u043b\u044f 2D-\u043c\u0430\u0442\u0440\u0438\u0446 (~35% \u043a \u0441\u043a\u043e\u0440\u043e\u0441\u0442\u0438 NanoGPT).\n    1D/\u0441\u043a\u0430\u043b\u044f\u0440\u044b/\u044d\u043c\u0431\u0435\u0434\u0434\u0438\u043d\u0433/pos \u043e\u0441\u0442\u0430\u044e\u0442\u0441\u044f \u043d\u0430 Adam (\u0441\u0442\u0430\u043d\u0434\u0430\u0440\u0442\u043d\u0430\u044f \u043f\u0440\u0430\u043a\u0442\u0438\u043a\u0430).\"\"\"\n    NS_COEF = (3.4445, -4.7750, 2.0315)\n    def __init__(self, params, lr=6e-4, mulr=0.02, mom=0.95, wd=0.1, ns=5, eps=1e-8):\n        self.p = params; self.lr0, self.mulr, self.mom, self.wd, self.ns, self.eps = lr, mulr, mom, wd, ns, eps\n        self.muon_keys = {k for k, v in params.d.items()\n                          if v.ndim == 2 and min(v.shape) >= 16 and k not in (\"E\", \"U\", \"pos\")}\n        self.adam_keys = set(params.d) - self.muon_keys\n        self.mb = {k: np.zeros_like(v) for k, v in params.d.items() if k in self.muon_keys}\n        self.m = {k: np.zeros_like(params.d[k]) for k in self.adam_keys}\n        self.v = {k: np.zeros_like(params.d[k]) for k in self.adam_keys}\n        self.t = 0\n        print(f\"[MuonW] muon: {sorted(self.muon_keys)[:4]}... ({len(self.muon_keys)} \u0448\u0442), adam: {len(self.adam_keys)} \u0448\u0442\", flush=True)\n    @staticmethod\n    def _ns5(G, steps, eps):\n        a, b, c = MuonW.NS_COEF\n        X = G / (np.linalg.norm(G.astype(np.float64)) + eps).astype(np.float32)\n        tr = X.shape[0] > X.shape[1]\n        if tr: X = X.T.copy()\n        for _ in range(steps):\n            A = X @ X.T\n            B = b * A + c * (A @ A)\n            X = a * X + B @ X\n        if tr: X = X.T\n        return X\n    def step(self, lr):\n        self.t += 1\n        for k in self.muon_keys:\n            g = self.p.g[k]\n            self.mb[k] = self.mom * self.mb[k] + (1 - self.mom) * g\n            gu = g * (1 - self.mom) + self.mom * self.mb[k]          # nesterov\n            o = self._ns5(gu, self.ns, self.eps)\n            scale = max(1.0, o.shape[0] / o.shape[1]) ** 0.5\n            self.p.d[k] -= self.mulr * scale * o + self.wd * lr * self.p.d[k]\n        for k in self.adam_keys:\n            g = self.p.g[k]\n            self.m[k] = 0.9 * self.m[k] + 0.1 * g\n            self.v[k] = 0.95 * self.v[k] + 0.05 * g * g\n            mh = self.m[k] / (1 - 0.9 ** self.t)\n            vh = self.v[k] / (1 - 0.95 ** self.t)\n            self.p.d[k] -= lr * (mh / (np.sqrt(vh) + self.eps) + self.wd * self.p.d[k])\n        self.p.zero()\n\n\n# ---------------------------------------------------------------- KDA-lite delta-\u043c\u0438\u043a\u0441\u0435\u0440\n# S_0 = 0;  p = S\u00b7k;  u = v \u2212 p;  S \u2190 Diag(a)\u00b7S + \u03b2\u00b7u\u2297k;  o = a\u2299p + \u03b2\u00b7u\u00b7\u2016k\u2016\u00b2   [k \u2261 v \u2261 y]\ndef delta_mix(X, th, sc, braw, Wg):            # X (B,T,D) \u2192 (Y, cache); k = x/\u2016x\u2016, v = x\n    a = 1.0 / (1.0 + np.exp(-th)); beta = 1.0 / (1.0 + np.exp(-braw))\n    B, T, D = X.shape\n    S = np.zeros((B, D, D), X.dtype)\n    Y = np.empty_like(X)\n    KN = np.empty((B, T), X.dtype)                # \u2016x_t\u2016\n    GS = np.empty((B, T, D), X.dtype)\n    P = np.empty_like(X); U = np.empty_like(X); N2 = np.empty((B, T), X.dtype)\n    KK = np.empty_like(X)\n    for t in range(T):                            # KDA-update, \u0441\u043e\u0441\u0442\u043e\u044f\u043d\u0438\u0435 \u043d\u0435 \u0441\u043a\u043b\u0430\u0434\u0438\u0440\u0443\u0435\u043c (backward replay)\n        v = X[:, t]\n        kn = np.sqrt((v * v).sum(-1)) + 1e-8\n        k = v / kn[:, None]\n        p_ = np.einsum('bij,bj->bi', S, k)\n        u = v - p_\n        n2 = (k * k).sum(-1)                      # \u22481\n        S = a[None, :, None] * S + beta * u[:, :, None] * k[:, None, :]\n        o = a[None, :] * p_ + beta * u * n2[:, None]\n        P[:, t] = p_; U[:, t] = u; N2[:, t] = n2; KK[:, t] = k; KN[:, t] = kn\n        Gt = 1.0 / (1.0 + np.exp(-(v @ Wg)))       # KDA output gate \u03c3(xW_g), \u043a\u0430\u043d\u0430\u043b\u044c\u043d\u044b\u0439\n        Y[:, t] = o * sc * Gt\n        GS[:, t] = Gt\n    return Y, (X, KK, KN, P, U, N2, S, a, beta, sc, GS, Wg)\n\ndef delta_mix_bwd(c, dY):\n    X, KK, KN, P, U, N2, Sfin, a, beta, sc, GS, Wg = c\n    B, T, D = X.shape\n    dX = np.zeros_like(X); dth = np.zeros(D, X.dtype); dbeta = 0.0\n    dWg = np.zeros_like(Wg)\n    dO_full = dY * sc * GS\n    dsc = (dY * (a[None, None, :] * P + beta * U * N2[:, :, None]) * GS).sum((0, 1))\n    dO = dO_full\n    G = np.zeros((B, D, D), X.dtype)              # dL/dS_t\n    da = np.zeros(D, X.dtype)\n    Scur = Sfin\n    for t in range(T - 1, -1, -1):\n        v = X[:, t]; k = KK[:, t]; kn = KN[:, t]\n        u = U[:, t]\n        S = (Scur - beta * u[:, :, None] * k[:, None, :]) / a[None, :, None]   # S_{t-1} \u0442\u043e\u0447\u043d\u043e\n        p = P[:, t]; n2 = N2[:, t]\n        dOt = dO[:, t]\n        da += (dOt * p).sum(0)\n        dbeta += float((dOt * u * n2[:, None]).sum())\n        dn2 = (dOt * (beta * u)).sum(-1)\n        da += (G * S).sum((0, 2))\n        dbeta += float((G * (u[:, :, None] * k[:, None, :])).sum())\n        du = dOt * beta * n2[:, None] + beta * np.einsum('bij,bj->bi', G, k)\n        dp = dOt * a[None, :] - du\n        dk = 2.0 * k * dn2[:, None] + beta * np.einsum('bij,bi->bj', G, u)\n        dk_tot = dk + np.einsum('bij,bi->bj', S, dp)\n        # \u0432\u044b\u0445\u043e\u0434\u043d\u043e\u0439 \u0433\u0435\u0439\u0442: d/dy_raw = y\u014d\u00b7sc\u00b7o\u00b7g(1\u2212g); \u0432 X-\u0442\u043e\u043a\u0435\u043d \u0447\u0435\u0440\u0435\u0437 Wg\n        o_t = a[None, :] * p + beta * u * n2[:, None]\n        dgr = dY[:, t] * sc * o_t * GS[:, t] * (1 - GS[:, t])\n        dWg += (v.T @ dgr)                        # v = x_t \u0441\u044b\u0440\u043e\u0439 (B,D)\u00b7(B,D)\n        dX[:, t] += dgr @ Wg.T\n        dX[:, t] += du + (dk_tot - k * (dk_tot * k).sum(-1, keepdims=True)) / kn[:, None]\n        G = a[None, :, None] * G + np.einsum('bi,bj->bij', dp, k)\n        Scur = S\n    dth = da * a * (1 - a)\n    dbraw = np.array(dbeta * beta * (1 - beta), X.dtype)\n    return dX, dth, dsc, dbraw, dWg\n\n# ---------------------------------------------------------------- MoE-FFN (DeepSeek-style, aux-loss-free)\nMOE_STATS = {}                                   # running loads \u0434\u043b\u044f bias-\u0430\u043f\u0434\u0435\u0439\u0442\u0430 (\u043d\u0435 \u0433\u0440\u0430\u0434\u0438\u0435\u043d\u0442\u043d\u043e)\n\ndef moe_ffn(X, Wr, be, W1, W2, key):           # X (B,T,D); W1 (E,D,F2); W2 (E,F2,D)\n    B, T, D = X.shape; E = Wr.shape[1]; F2 = W1.shape[2]\n    s = X @ Wr                                   # (B,T,E) affinity (\u0441\u044b\u0440\u044b\u0435)\n    sel = (s + be).argmax(-1)                    # hard top-1 \u0441 bias-\u043a\u043e\u0440\u0440\u0435\u043a\u0446\u0438\u0435\u0439 (aux-loss-free)\n    sg = 1.0 / (1.0 + np.exp(-s))\n    gv = np.take_along_axis(sg, sel[..., None], -1)[..., 0]      # gate = \u03c3(s_sel), bias \u0432 \u0432\u044b\u0431\u043e\u0440 \u041d\u0415 \u0432 \u0437\u043d\u0430\u0447\u0435\u043d\u0438\u0435\n    with np.errstate(over='ignore'):\n        pass\n    out = np.zeros_like(X)\n    for e in range(E):\n        m = sel == e\n        if not m.any(): continue\n        xe = X[m]\n        h = gelu(xe @ W1[e])[0] @ W2[e]\n        out[m] = gv[m, None] * h\n    loads = np.bincount(sel.reshape(-1), minlength=E) / float(B * T)\n    MOE_STATS.setdefault(key, np.full(E, 1.0 / E))\n    MOE_STATS[key] = 0.99 * MOE_STATS[key] + 0.01 * loads\n    return out, (X, Wr, be, W1, W2, sel, sg, gv, s)\n\ndef moe_ffn_bwd(c, dOut):\n    X, Wr, be, W1, W2, sel, sg, gv, s = c\n    B, T, D = X.shape; E = Wr.shape[1]; F2 = W1.shape[2]\n    dX = np.zeros_like(X); dWr = np.zeros_like(Wr)\n    dW1 = np.zeros_like(W1); dW2 = np.zeros_like(W2)\n    ds = np.zeros_like(s)\n    for e in range(E):\n        m = sel == e\n        if not m.any(): continue\n        xe = X[m]; dO = dOut[m]\n        dgv = (dO * (gelu(xe @ W1[e])[0] @ W2[e])).sum(-1)         # (N,)\n        dh = dO * gv[m, None]\n        z1, cg = gelu(xe @ W1[e])\n        dz1 = gelu_bwd(cg, dh @ W2[e].T)\n        dW2[e] += z1.T @ dh\n        dW1[e] += xe.T @ dz1\n        dX[m] += dz1 @ W1[e].T\n        dsel = dgv * gv[m] * (1 - gv[m])\n        ds[..., e][m] += dsel\n        # gate \u03c3(s_sel): \u0432\u043a\u043b\u0430\u0434 \u0432 \u043e\u0441\u0442\u0430\u043b\u044c\u043d\u044b\u0435 \u044d\u043a\u0441\u043f\u0435\u0440\u0442\u044b = 0 (top-1 hard, \u043a\u0430\u043a \u0438 \u0432\u044b\u0431\u043e\u0440)\n    dWr = X.reshape(-1, D).T @ ds.reshape(-1, E)\n    dX += ds @ Wr.T\n    dbe = ds.sum((0, 1)) * 0.0                     # bias \u043e\u0431\u043d\u043e\u0432\u043b\u044f\u0435\u0442\u0441\u044f running-\u0441\u0442\u0430\u0442\u0438\u0441\u0442\u0438\u043a\u043e\u0439\n    return dX, dWr, dbe, dW1, dW2\n\ndef main():\n    args = ap_parse()\n    tr = hnp.load(f\"{ROOT}/{args.data}/train.npy\").astype(hnp.int64)\n    va = hnp.load(f\"{ROOT}/{args.data}/val.npy\").astype(hnp.int64)\n    V = json.load(open(f\"{ROOT}/{args.data}/meta.json\"))[\"vocab\"]\n    rng = hnp.random.default_rng(args.seed)\n    rng_neg = hnp.random.default_rng(args.seed * 1000003 + 17) if getattr(args, 'negrng', 0) else rng\n    assert not (args.erank > 0 and args.mtp > 0), \"erank+mtp \u043f\u043e\u043a\u0430 \u043d\u0435 \u043a\u043e\u043c\u0431\u0438\u043d\u0438\u0440\u0443\u044e\u0442\u0441\u044f\"\n    model = NanoGPT(V, D=args.dim, L=args.layers, ff=args.ff, kind=args.kind, adr_kf=args.adr, moe_e=args.moe, mtp_w=args.mtp, erank=args.erank, kronfc=args.kronfc)\n    teacher = None\n    if args.distill:\n        teacher = NanoGPT(V)\n        td = hnp.load(args.distill)\n        for k in teacher.p.d:\n            if k in td.files: teacher.p.d[k][...] = td[k]\n        print(f\"[{args.tag}] \u0443\u0447\u0438\u0442\u0435\u043b\u044c {args.distill} \u0437\u0430\u0433\u0440\u0443\u0436\u0435\u043d\", flush=True)\n    if args.initckpt:\n        d0 = hnp.load(args.initckpt)\n        for k in model.p.d:\n            if k in d0.files: model.p.d[k][...] = d0[k]\n        print(f\"[{args.tag}] init from {args.initckpt}\", flush=True)\n    nparams = sum(v.size for v in model.p.d.values())\n    print(f\"[{args.tag}] nano-{args.kind} params={nparams:,} backend={'cupy-gpu' if on_gpu else 'numpy'}\", flush=True)\n    N0_TRUNK = init_norms(model.p.d) if getattr(args, \"trunknorm\", 0) else None\n    if args.opt == \"muon\": opt = MuonW(model.p, lr=args.lr, mulr=args.mulr)\n    else: opt = AdamW(model.p, lr=args.lr, wdmask=bool(args.wdmask))\n\n    ew = None\n    if args.wema > 0:\n        ew = {k: np.zeros_like(v) for k, v in model.p.d.items()}\n        ew_corr = 1.0\n        print(f\"[{args.tag}] weight-EMA decay={args.wema}: \u0434\u0432\u043e\u0439\u043d\u0430\u044f \u043e\u0446\u0435\u043d\u043a\u0430 val (live + ema)\", flush=True)\n\n    # ---- sampled softmax: \u043f\u0440\u0435\u0434\u043b\u043e\u0436\u0435\u043d\u0438\u0435 q = unigram(train), \u043a\u043e\u0440\u0440\u0435\u043a\u0446\u0438\u044f Q_c = 1\u2212(1\u2212q)^K (with-replacement)\n    ssq = None\n    if args.ssk > 0:\n        assert teacher is None, \"ssk: distill \u043f\u043e\u043a\u0430 \u043d\u0435 \u043a\u043e\u043c\u0431\u0438\u043d\u0438\u0440\u0443\u0435\u0442\u0441\u044f\"\n        assert args.mtp == 0, \"ssk: mtp \u043f\u043e\u043a\u0430 \u043d\u0435 \u043a\u043e\u043c\u0431\u0438\u043d\u0438\u0440\u0443\u0435\u0442\u0441\u044f\"\n        cnt = hnp.bincount(tr, minlength=V).astype(hnp.float64) + 1.0\n        ssq = cnt ** args.ssalpha\n        ssq = ssq / ssq.sum()\n        print(f\"[{args.tag}] sampled softmax: K={args.ssk} \u043d\u0435\u0433\u0430\u0442\u0438\u0432\u043e\u0432/\u0448\u0430\u0433, unigram-\u043f\u0440\u0435\u0434\u043b\u043e\u0436\u0435\u043d\u0438\u0435, \"\n              f\"logQ-\u043a\u043e\u0440\u0440\u0435\u043a\u0446\u0438\u044f; \u0432\u0430\u043b \u2014 \u043f\u043e\u043b\u043d\u044b\u0439 softmax\", flush=True)\n\n    def batch(ids, rr):\n        st = rr.integers(0, len(ids) - args.ctx - 1, size=args.batch)\n        x = hnp.stack([ids[s:s + args.ctx] for s in st]); y = hnp.stack([ids[s + 1:s + args.ctx + 1] for s in st])\n        return to_dev(x), to_dev(y)\n\n    def vloss(iters=6):\n        rr = hnp.random.default_rng(1234); tot = 0.0\n        for _ in range(iters):\n            x, y = batch(va, rr)\n            H, _ = model.forward(x)\n            l, _ = softmax_ce(model.logits(H), y, destroy=True)\n            tot += l\n        return tot / iters\n\n    os.makedirs(f\"{ROOT}/results\", exist_ok=True)\n    logf = open(f\"{ROOT}/results/run_{args.tag}.jsonl\", \"w\")\n    t0 = time.time(); toks = 0\n    for step in range(args.steps):\n        lr = args.lr * min((step + 1) / args.warmup,\n              0.1 + 0.45 * (1 + math.cos(math.pi * max(0, step - args.warmup) / max(1, args.steps - args.warmup))))\n        _t = time.process_time_ns()\n        x, y = batch(tr, rng)\n        _t = _tick(\"batch\", _t)\n        H, caches = model.forward(x)\n        _t = _tick(\"fwd\", _t)\n        head_cache = None\n        use_ss = ssq is not None and step < args.steps * (1.0 - args.ssfull)\n        if use_ss:\n            neg = rng_neg.choice(V, size=args.ssk, replace=True, p=ssq)\n            yh = asnumpy(y)\n            cand = hnp.union1d(hnp.unique(yh), neg).astype(hnp.int64)\n            logQ = hnp.log1p(-hnp.power(1.0 - ssq[cand], args.ssk))    # log(1\u2212(1\u2212q)^K)\n            logQ[hnp.isin(cand, yh)] = 0.0                             # \u0446\u0435\u043b\u0438 \u0432\u0441\u0435\u0433\u0434\u0430 \u0432 \u043d\u0430\u0431\u043e\u0440\u0435\n            cand_d, logQ_d = to_dev(cand), to_dev(logQ)\n            Ec0 = (model.p.d[\"U\"][cand_d] @ model.p.d[\"P\"]) if model.erank > 0 else None\n            loss, (dz, cand, Ec) = sampled_ce(H, y, model.p.d[\"E\"] if model.erank == 0 else Ec0, cand_d, logQ_d, Ec=Ec0)\n            head_cache = (cand, dz, Ec)\n            dz = None\n        else:\n            lg = model.logits(H)\n            loss, dz = softmax_ce(lg, y, destroy=(teacher is None))\n        _t = _tick(\"head_fwd\", _t)\n        if teacher is not None:\n            Ht, _ = teacher.forward(x)\n            lgt = Ht @ teacher.p.d[\"E\"].T\n            zt = lgt / args.dtemp; q = np.exp(zt - zt.max(-1, keepdims=True)); q /= q.sum(-1, keepdims=True)\n            zs = lg / args.dtemp; ps = np.exp(zs - zs.max(-1, keepdims=True)); ps /= ps.sum(-1, keepdims=True)\n            n = lg.shape[0] * lg.shape[1]\n            kl = float((q * (np.log(q + 1e-12) - np.log(ps + 1e-12))).sum() / n)\n            dz = dz + args.dkl * (args.dtemp ** 2) * (ps - q) / n\n            loss = loss + args.dkl * kl\n        dH_aux = None\n        if args.mtp > 0:\n            y2 = np.concatenate([y[:, 1:], x[:, -1:]], axis=1)     # t+2 \u0446\u0435\u043b\u044c (\u043f\u043e\u0441\u043b\u0435\u0434\u043d\u0438\u0439 \u2014 \u0444\u0438\u043a\u0442\u0438\u0432, \u0432\u0435\u0441 0)\n            zn, clnm = layernorm(H, model.p.d[\"lnm_g\"], model.p.d[\"lnm_b\"])\n            zm, cwm = linear(zn, model.p.d[\"Wmtp\"])\n            lg2 = model.logits(zm)\n            loss2, dz2 = softmax_ce(lg2, y2)\n            dz2[:, -1] = 0                                        # \u0444\u0438\u043a\u0442\u0438\u0432\u043d\u0430\u044f \u0446\u0435\u043b\u044c\n            loss = loss + args.mtp * loss2\n            dz_m = dz2 @ model.p.d[\"E\"]                           # \u0447\u0435\u0440\u0435\u0437 tied-E\n            # \u0442\u043e\u0447\u043d\u0435\u0435: lg2 = zm @ E.T \u2192 dE += dz2\u1d40\u00b7zm; dzm = dz2 @ E\n            model.p.g[\"E\"] += args.mtp * dz2.reshape(-1, model.V).T @ zm.reshape(-1, model.D)\n            dzm = dz2 @ model.p.d[\"E\"]\n            dzn, dWmtp = linear_bwd(cwm, args.mtp * dzm); model.p.g[\"Wmtp\"] += dWmtp\n            dH_aux, dgm, dbm = layernorm_bwd(clnm, dzn)\n            model.p.g[\"lnm_g\"] += dgm; model.p.g[\"lnm_b\"] += dbm\n        model.backward(H, caches, dz, dH_aux=dH_aux, head_cache=head_cache)\n        _t = _tick(\"bwd_total\", _t)\n        if args.moe > 1:\n            for i in range(model.L):\n                be = model.p.d[f\"b{i}.be\"]\n                be += 0.02 * (0.25 - MOE_STATS[i])         # bias \u2191 \u043d\u0435\u0434\u043e\u0433\u0440\u0443\u0436\u0435\u043d\u043d\u044b\u043c \u044d\u043a\u0441\u043f\u0435\u0440\u0442\u0430\u043c\n        opt.step(lr)\n        _t = _tick(\"opt\", _t)\n        if ew is not None:\n            d = args.wema\n            for k in ew:\n                ew[k] = d * ew[k] + (1.0 - d) * model.p.d[k]\n            ew_corr = 1.0 - d ** (step + 1)\n        toks += x.size\n        if step % args.eval_every == 0 or step == args.steps - 1:\n            vl = vloss(); el = time.time() - t0\n            rec = dict(step=step, train_loss=round(float(loss), 4), val_loss=round(float(vl), 4),\n                       val_ppl=round(float(math.exp(vl)), 2), wall=round(el, 1),\n                       tok_s=round(toks / el, 1), tokens=toks)\n            if ew is not None:\n                bk = {k: model.p.d[k].copy() for k in ew}\n                for k in ew: model.p.d[k][...] = ew[k] / ew_corr      # bias-\u043a\u043e\u0440\u0440\u0435\u043a\u0446\u0438\u044f EMA\n                vle = vloss()\n                for k in ew: model.p.d[k][...] = bk[k]\n                rec[\"val_ppl_ema\"] = round(float(math.exp(vle)), 2)\n            if N0_TRUNK is not None: rec[\"wn\"] = round(trunk_ratio(model.p.d, N0_TRUNK), 3)\n            print(json.dumps(rec), flush=True); logf.write(json.dumps(rec) + \"\\n\"); logf.flush()\n    save_npz(f\"{ROOT}/results/ckpt_{args.tag}.npz\", model.p.d)\n    if ew is not None:\n        save_npz(f\"{ROOT}/results/ckpt_{args.tag}_ema.npz\", {k: ew[k] / ew_corr for k in ew})\n    print(\"SUMMARY \" + json.dumps(dict(tag=args.tag, kind=args.kind, params=nparams,\n        tokens=toks, wall_s=round(time.time() - t0, 1), tok_s=round(toks / (time.time() - t0), 1),\n        final_val_loss=rec[\"val_loss\"], final_val_ppl=rec[\"val_ppl\"])), flush=True)\n    if os.environ.get(\"NANOLC_PROF\"):\n        tot = sum(np.median(v) for v in PROF.values()) / 1e6\n        print(\"PROF \u043c\u0435\u0434\u0438\u0430\u043d\u044b ms/\u0448\u0430\u0433:\", flush=True)\n        for k in sorted(PROF, key=lambda k: -np.median(PROF[k])):\n            v = np.median(PROF[k]) / 1e6\n            print(f\"  {k:12s} {v:8.1f} ({100*v/tot:.1f}%)\", flush=True)\n        print(f\"  {'\u0418\u0422\u041e\u0413\u041e':12s} {tot:8.1f}\", flush=True)\n\ndef ap_parse():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--kind\", choices=[\"attn\", \"ema\", \"hybrid\", \"delta\"], required=True)\n    ap.add_argument(\"--opt\", choices=[\"adam\", \"muon\"], default=\"adam\")\n    ap.add_argument(\"--wdmask\", type=int, default=0, help=\"AdamW: wd \u0442\u043e\u043b\u044c\u043a\u043e \u043d\u0430 \u0441\u043a\u0440\u044b\u0442\u044b\u0445 2D-\u043c\u0430\u0442\u0440\u0438\u0446\u0430\u0445\")\n    ap.add_argument(\"--wema\", type=float, default=0.0, help=\"decay EMA \u0432\u0435\u0441\u043e\u0432 \u0434\u043b\u044f \u043e\u0446\u0435\u043d\u043a\u0438 (0=\u0432\u044b\u043a\u043b)\")\n    ap.add_argument(\"--mulr\", type=float, default=0.02)\n    ap.add_argument(\"--moe\", type=int, default=1, help=\"E \u044d\u043a\u0441\u043f\u0435\u0440\u0442\u043e\u0432 top-1 (DeepSeek aux-loss-free); 1 = dense FFN\")\n    ap.add_argument(\"--kronfc\", type=int, default=0, help=\"K \u043a\u0440\u043e\u043d\u0435\u043a\u0435\u0440-\u0447\u043b\u0435\u043d\u043e\u0432 \u0432 FFN fc1/fc2 (0=dense)\")\n    ap.add_argument(\"--mtp\", type=float, default=0.0, help=\"\u0432\u0435\u0441 MTP-aux \u0433\u043e\u043b\u043e\u0432\u044b t+2 (DeepSeek); 0 \u2014 \u0432\u044b\u043a\u043b\")\n    ap.add_argument(\"--dim\", type=int, default=192); ap.add_argument(\"--layers\", type=int, default=4)\n    ap.add_argument(\"--ff\", type=int, default=576)\n    ap.add_argument(\"--distill\", default=None, help=\"ckpt \u0443\u0447\u0438\u0442\u0435\u043b\u044f (fp32 ema 192x4) \u0434\u043b\u044f KL-\u0434\u0438\u0441\u0442\u0438\u043b\u043b\u044f\u0446\u0438\u0438\")\n    ap.add_argument(\"--dtemp\", type=float, default=2.0); ap.add_argument(\"--dkl\", type=float, default=0.5)\n    ap.add_argument(\"--tag\", required=True)\n    ap.add_argument(\"--steps\", type=int, default=500)\n    ap.add_argument(\"--batch\", type=int, default=24)\n    ap.add_argument(\"--ctx\", type=int, default=96)\n    ap.add_argument(\"--adr\", type=float, default=None)\n    ap.add_argument(\"--lr\", type=float, default=6e-4)\n    ap.add_argument(\"--seed\", type=int, default=42)\n    ap.add_argument(\"--warmup\", type=int, default=60)\n    ap.add_argument(\"--eval_every\", type=int, default=50)\n    ap.add_argument(\"--initckpt\", default=None)\n    ap.add_argument(\"--data\", default=\"data/prep\")\n    ap.add_argument(\"--ssk\", type=int, default=0, help=\"sampled softmax: K \u043d\u0435\u0433\u0430\u0442\u0438\u0432\u043e\u0432/\u0448\u0430\u0433 (0 = \u043f\u043e\u043b\u043d\u044b\u0439 CE)\")\n    ap.add_argument(\"--ssfull\", type=float, default=0.0, help=\"\u0434\u043e\u043b\u044f \u0444\u0438\u043d\u0430\u043b\u044c\u043d\u044b\u0445 \u0448\u0430\u0433\u043e\u0432 \u0441 \u043f\u043e\u043b\u043d\u044b\u043c CE (\u0430\u043d\u043d\u0438\u043b)\")\n    ap.add_argument(\"--ssalpha\", type=float, default=1.0, help=\"\u0441\u0442\u0435\u043f\u0435\u043d\u044c \u0441\u0433\u043b\u0430\u0436\u0438\u0432\u0430\u043d\u0438\u044f unigram-\u043f\u0440\u0435\u0434\u043b\u043e\u0436\u0435\u043d\u0438\u044f (word2vec 0.75)\")\n    ap.add_argument(\"--negrng\", type=int, default=0, help=\"1 = \u043e\u0442\u0434\u0435\u043b\u044c\u043d\u044b\u0439 rng-\u043f\u043e\u0442\u043e\u043a \u0434\u043b\u044f \u043d\u0435\u0433\u0430\u0442\u0438\u0432\u043e\u0432 (\u043f\u0430\u0440\u043d\u043e\u0441\u0442\u044c \u043f\u043e\u0440\u044f\u0434\u043a\u0430 \u0434\u0430\u043d\u043d\u044b\u0445 \u043c\u0435\u0436\u0434\u0443 \u043a\u043e\u043d\u0444\u0438\u0433\u0430\u043c\u0438)\")\n    ap.add_argument(\"--trunknorm\", type=int, default=0, help=\"1 = \u043b\u043e\u0433 \u2016W_trunk\u2016/\u2016W\u2080\u2016 \u0432 jsonl (\u0430\u0432\u0442\u043e-DQ \u043c\u0451\u0440\u0442\u0432\u044b\u0445 \u043f\u0440\u043e\u0433\u043e\u043d\u043e\u0432)\")\n    ap.add_argument(\"--erank\", type=int, default=0, help=\"\u043d\u0438\u0437\u043a\u043e\u0440\u0430\u043d\u0433\u043e\u0432\u044b\u0439 tied-\u044d\u043c\u0431\u0435\u0434\u0434\u0438\u043d\u0433 U(V,r)@P(r,D); 0 = \u0432\u044b\u043a\u043b\")\n    return ap.parse_args()\n\nif __name__ == \"__main__\":\n    main()\n")
print('nano_lc_kg.py ok')

In [ ]:
open(f'{BASE}/train_torch.py','w').write("#!/usr/bin/env python3\n# -*- coding: utf-8 -*-\n\"\"\"train_torch.py \u2014 torch-\u0434\u0432\u0438\u0436\u043e\u043a \u0442\u0443\u0440\u043d\u0438\u0440\u0430 \u0441 \u0441\u0435\u043c\u0430\u043d\u0442\u0438\u043a\u043e\u0439 nano_lc 1:1 (\u043d\u0430\u0441\u043a\u043e\u043b\u044c\u043a\u043e \u043f\u043e\u0437\u0432\u043e\u043b\u044f\u0435\u0442 fp32-\u043f\u043e\u0440\u044f\u0434\u043e\u043a \u0440\u0435\u0434\u0443\u043a\u0446\u0438\u0439).\n\u0421\u0435\u043c\u0430\u043d\u0442\u0438\u043a\u0430 (\u043d\u0430\u043c\u0435\u0440\u0435\u043d\u043d\u043e \u043f\u0440\u043e\u0434\u0443\u0431\u043b\u0438\u0440\u043e\u0432\u0430\u043d\u0430, \u041d\u0415 \u0438\u043c\u043f\u043e\u0440\u0442\u0438\u0440\u0443\u0435\u0442\u0441\u044f \u0438\u0437 leancore_torch.py \u2014 \u0442\u0430\u043c lr-\u0441\u0435\u043c\u0430\u043d\u0442\u0438\u043a\u0430\n\u0438 \u0431\u0435\u0442\u044b MuonW \u0440\u0430\u0441\u0445\u043e\u0434\u044f\u0442\u0441\u044f \u0441 \u043e\u0440\u0438\u0433\u0438\u043d\u0430\u043b\u043e\u043c):\n  \u0431\u043b\u043e\u043a (ema):  x += ln1\u2192EMA(a=\u03c3(th), y=sc\u00b7h)@Wm \u2192 +x \u2192 ln2 \u2192 fc1\u2192gelu(tanh)\u2192fc2  [ADR off]\n  \u0437\u0430\u0442\u0435\u043c lnf; \u0433\u043e\u043b\u043e\u0432\u0430 tied E. MuonW: muon 2D-\u043c\u0430\u0440\u0448\u0440\u0443\u0442\u044b (.Wm/.fc1/.fc2, ndim2, min\u226516, \u043a\u0440\u043e\u043c\u0435 E/U/pos),\n  \u043e\u0431\u043d\u043e\u0432\u043b\u0435\u043d\u0438\u0435 p\u00b7=(1\u2212wd\u00b7lr); p\u2212=mulr\u00b7scale\u00b7NS5(u) (mulr \u0411\u0415\u0417 \u0443\u043c\u043d\u043e\u0436\u0435\u043d\u0438\u044f \u043d\u0430 lr \u2014 \u043a\u0430\u043a nano_lc);\n  adam-\u0432\u0435\u0442\u043a\u0430: b1=0.9, b2=0.95, eps=1e-8, wd=0.1\u00b7lr\u00b7p.\n  LR:  lr\u00b7min((s+1)/warmup, 0.1+0.45(1+cos(\u03c0\u00b7max(0,s\u2212warmup)/(steps\u2212warmup)))).\n  SSK:  \u043d\u0435\u0433\u0430\u0442\u0438\u0432\u044b unigram cnt**ssalpha (host numpy!), \u043a\u0430\u043d\u0434\u0438\u0434\u0430\u0442\u044b = union(\u0446\u0435\u043b\u0438, neg), logQ-\u043a\u043e\u0440\u0440\u0435\u043a\u0446\u0438\u044f\n        log(1\u2212(1\u2212q)^K), \u0446\u0435\u043b\u0438 \u043e\u0431\u043d\u0443\u043b\u044f\u044e\u0442 \u043a\u043e\u0440\u0440\u0435\u043a\u0446\u0438\u044e; ssfull \u2014 \u0434\u043e\u043b\u044f \u0444\u0438\u043d\u0430\u043b\u044c\u043d\u044b\u0445 \u0448\u0430\u0433\u043e\u0432 \u0441 \u043f\u043e\u043b\u043d\u044b\u043c CE.\n  RNG:  \u0434\u0430\u043d\u043d\u044b\u0435 \u2014 host numpy default_rng(seed); \u043d\u0435\u0433\u0430\u0442\u0438\u0432\u044b \u2014 negrng=1 \u2192 \u043e\u0442\u0434\u0435\u043b\u044c\u043d\u044b\u0439 \u043f\u043e\u0442\u043e\u043a\n        default_rng(seed\u00b71000003+17); \u0432\u0430\u043b \u2014 default_rng(1234) \u0444\u0438\u043a\u0441\u0438\u0440\u043e\u0432\u0430\u043d.\n  \u0418\u041d\u0418\u0422: torch.manual_seed(0x1EA7) \u2014 \u0424\u0418\u041a\u0421\u0418\u0420\u041e\u0412\u0410\u041d, \u043d\u0435 \u0437\u0430\u0432\u0438\u0441\u0438\u0442 \u043e\u0442 --seed (\u0437\u0435\u0440\u043a\u0430\u043b\u043e crc32-\u0438\u043d\u0438\u0446\u0438\u0430\u043b\u0438\u0437\u0430\u0446\u0438\u0438;\n        \u043f\u0430\u0440\u043d\u043e\u0441\u0442\u044c \u043a\u043e\u043d\u0444\u0438\u0433\u043e\u0432 = \u043e\u0434\u0438\u043d\u0430\u043a\u043e\u0432\u044b\u0439 \u0438\u043d\u0438\u0442 \u0443 \u0432\u0441\u0435\u0445).\n\u041f\u043e\u0442\u043e\u043a: host-numpy \u0434\u0430\u043d\u043d\u044b\u0435 \u2192 gpu-\u0442\u0435\u043d\u0437\u043e\u0440\u044b; autograd \u0447\u0435\u0440\u0435\u0437 E[cand] \u0441\u0430\u043c \u0440\u0430\u0437\u0431\u0440\u0430\u0441\u044b\u0432\u0430\u0435\u0442 \u0433\u0440\u0430\u0434\u0438\u0435\u043d\u0442 \u0433\u043e\u043b\u043e\u0432\u044b.\n\"\"\"\nimport os, sys, json, math, time, argparse\nimport numpy as hnp\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nINIT_SEED = 0x1EA7          # \u0444\u0438\u043a\u0441\u0438\u0440\u043e\u0432\u0430\u043d\u043d\u044b\u0439 \u0438\u043d\u0438\u0442 \u0434\u043b\u044f \u0412\u0421\u0415\u0425 \u043a\u043e\u043d\u0444\u0438\u0433\u043e\u0432 (\u043f\u0430\u0440\u043d\u043e\u0441\u0442\u044c \u0431\u0440\u0435\u043a\u0435\u0442\u0430)\nf32 = torch.float32\n\n\ndef np_paired_init(model, hnp, zlib):\n    \"\"\"\u0411\u0438\u0442\u043e\u0432\u043e-\u043f\u0430\u0440\u043d\u044b\u0439 \u0438\u043d\u0438\u0442 \u0441 numpy-\u0442\u0440\u0435\u043d\u0435\u0440\u043e\u043c: \u0442\u043e\u0442 \u0436\u0435 default_rng(crc32(\u0438\u043c\u044f)) \u0438 \u0442\u0435 \u0436\u0435\n    \u0441\u043f\u0435\u0446-\u0438\u043d\u0438\u0442\u044b (ln=ones/zeros, th=0, sc=1). \u0411\u0435\u0437 \u044d\u0442\u043e\u0433\u043e torch \u0438 numpy \u0441\u0442\u0430\u0440\u0442\u043e\u0432\u0430\u043b\u0438 \u0438\u0437\n    \u0420\u0410\u0417\u041d\u042b\u0425 \u0441\u043b\u0443\u0447\u0430\u0439\u043d\u044b\u0445 \u0442\u043e\u0447\u0435\u043a \u0438 GPU-\u0433\u0435\u0439\u0442 \u0438\u0437\u043c\u0435\u0440\u044f\u043b \u0448\u0443\u043c \u0438\u043d\u0438\u0442\u0430, \u0430 \u043d\u0435 \u0440\u0430\u0441\u0445\u043e\u0436\u0434\u0435\u043d\u0438\u0435 \u0434\u0432\u0438\u0436\u043a\u043e\u0432\n    [\u0438\u0437\u043c\u0435\u0440\u0435\u043d\u043e \u043d\u0430 Kaggle: rel\u0394@40\u0448\u0430\u0433\u043e\u0432 = 2.0044% \u043f\u0440\u0438 \u043f\u043e\u0440\u043e\u0433\u0435 2.0% \u2014 \u0432\u043d\u0435\u0431\u0440\u0430\u043a\u043e\u0432\u043e\u0447\u043d\u043e].\"\"\"\n    import re\n    TABLE = {\"ln1.weight\": \"ln1g\", \"ln1.bias\": \"ln1b\", \"ln2.weight\": \"ln2g\", \"ln2.bias\": \"ln2b\",\n             \"th\": \"th\", \"sc\": \"sc\", \"Wm\": \"Wm\", \"fc1\": \"fc1\", \"fc2\": \"fc2\"}\n    CONST = {\"th\": 0.0, \"sc\": 1.0, \"ln1g\": 1.0, \"ln1b\": 0.0, \"ln2g\": 1.0, \"ln2b\": 0.0,\n             \"lnfg\": 1.0, \"lnfb\": 0.0}\n\n    def mapped(tname):\n        m = re.fullmatch(r\"blocks\\.(\\d+)\\.(.+)\", tname)\n        if m:\n            return f\"b{m.group(1)}.\" + TABLE[m.group(2)]\n        return {\"lnf.weight\": \"lnfg\", \"lnf.bias\": \"lnfb\"}.get(tname, tname)\n\n    with torch.no_grad():\n        for tname, p in model.named_parameters():\n            nname = mapped(tname)\n            if nname in CONST:\n                p.fill_(CONST[nname])\n                continue\n            rng = hnp.random.default_rng(zlib.crc32(nname.encode()) & 0xFFFFFFFF)\n            arr = rng.normal(0.0, 0.02, tuple(p.shape)).astype(hnp.float32)\n            p.copy_(torch.from_numpy(arr).to(p.dtype))\n\n\n# ---------------------------------------------------------------- Muon: NS5 fp32-\u0434\u043b\u044f-sm75/sm_60\ndef _ns_dtype():\n    if torch.cuda.is_available():\n        try:\n            if torch.cuda.get_device_capability(0) < (8, 0):\n                return torch.float32      # T4/P100 \u0430\u043f\u043f\u0430\u0440\u0430\u0442\u043d\u043e bf16 \u043d\u0435 \u0443\u043c\u0435\u044e\u0442\n        except Exception:\n            pass\n    return torch.bfloat16\n\n\ndef ns5(G, steps=5):\n    a, b, c = (3.4445, -4.7750, 2.0315)\n    X = G.to(_ns_dtype())\n    X = X / (X.norm() + 1e-7)\n    tr = G.size(0) > G.size(1)\n    if tr:\n        X = X.mT\n    for _ in range(steps):\n        A = X @ X.mT\n        B = b * A + c * (A @ A)\n        X = a * X + B @ X\n    if tr:\n        X = X.mT\n    return X.to(G.dtype)\n\n\n# ---------------------------------------------------------------- EMA-\u043c\u0438\u043a\u0441\u0435\u0440 (\u0437\u0430\u043c\u043a\u043d\u0443\u0442\u0430\u044f \u03a3-\u0444\u043e\u0440\u043c\u0430)\ndef ema_mix(X, th, sc):\n    B, T, D = X.shape\n    a = torch.sigmoid(th)\n    tt = torch.arange(T, device=X.device)\n    dd = (tt[:, None] - tt[None, :]).clamp(min=0).to(f32)\n    alog = torch.log(a.clamp_min(1e-20))\n    P = torch.exp(dd[:, :, None] * alog[None, None, :])\n    mask = (tt[:, None] >= tt[None, :]).to(f32)\n    M = P * mask[:, :, None] * (1 - a)[None, None, :]\n    H = torch.einsum('tkd,bkd->btd', M, X)\n    return H * sc\n\n\nclass Block(nn.Module):\n    def __init__(self, D, ff):\n        super().__init__()\n        self.ln1 = nn.LayerNorm(D); self.ln2 = nn.LayerNorm(D)\n        self.th = nn.Parameter(torch.zeros(D)); self.sc = nn.Parameter(torch.ones(D))\n        self.Wm = nn.Parameter(torch.empty(D, D)); nn.init.normal_(self.Wm, std=0.02)\n        self.fc1 = nn.Parameter(torch.empty(D, ff)); nn.init.normal_(self.fc1, std=0.02)\n        self.fc2 = nn.Parameter(torch.empty(ff, D)); nn.init.normal_(self.fc2, std=0.02)\n\n    def forward(self, x):\n        mix = ema_mix(self.ln1(x), self.th, self.sc) @ self.Wm\n        z = self.ln2(x + mix)\n        o2 = F.gelu(z @ self.fc1, approximate='tanh') @ self.fc2\n        return x + mix + o2\n\n\nclass LeanCore(nn.Module):\n    def __init__(self, V, D=192, L=4, ff=576):\n        super().__init__()\n        self.V, self.D = V, D\n        self.E = nn.Parameter(torch.empty(V, D)); nn.init.normal_(self.E, std=0.02)\n        self.pos = nn.Parameter(torch.empty(96, D)); nn.init.normal_(self.pos, std=0.02)\n        self.blocks = nn.ModuleList([Block(D, ff) for _ in range(L)])\n        self.lnf = nn.LayerNorm(D)\n\n    def forward(self, ids):\n        h = self.E[ids] + self.pos[:ids.shape[1]][None]\n        for b in self.blocks:\n            h = b(h)\n        return self.lnf(h)\n\n    def logits(self, h):\n        return h @ self.E.t()\n\n\nMUON_SKIP = (\"E\", \"U\", \"pos\")\n\n\ndef muon_split(model):\n    mu, ad = [], []\n    for n, p in model.named_parameters():\n        short = n.split(\".\")[-1]\n        if p.ndim == 2 and min(p.shape) >= 16 and short not in MUON_SKIP:\n            mu.append((n, p))\n        else:\n            ad.append((n, p))\n    return mu, ad\n\n\ndef trunk_ratio(model, n0):\n    tot = 0.0\n    for n, p in model.named_parameters():\n        short = n.split(\".\")[-1]\n        if p.ndim == 2 and min(p.shape) >= 16 and short not in MUON_SKIP:\n            v = p.detach().double().cpu()\n            tot += float((v * v).sum())\n    return tot ** 0.5 / n0\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    ap.add_argument(\"--steps\", type=int, default=500)\n    ap.add_argument(\"--eval_every\", type=int, default=100)\n    ap.add_argument(\"--lr\", type=float, default=6e-4)\n    ap.add_argument(\"--warmup\", type=int, default=60)\n    ap.add_argument(\"--mulr\", type=float, default=0.02)\n    ap.add_argument(\"--wd\", type=float, default=0.1)\n    ap.add_argument(\"--ssk\", type=int, default=0)\n    ap.add_argument(\"--ssfull\", type=float, default=0.0)\n    ap.add_argument(\"--ssalpha\", type=float, default=1.0)\n    ap.add_argument(\"--seed\", type=int, default=42)\n    ap.add_argument(\"--negrng\", type=int, default=0)\n    ap.add_argument(\"--trunknorm\", type=int, default=0)\n    ap.add_argument(\"--batch\", type=int, default=24)\n    ap.add_argument(\"--ctx\", type=int, default=96)\n    ap.add_argument(\"--data\", default=\"data/prep\")\n    ap.add_argument(\"--tag\", required=True)\n    ap.add_argument(\"--outdir\", default=\"results\")\n    ap.add_argument(\"--saveckpt\", type=int, default=0)\n    args = ap.parse_args()\n\n    dev = \"cuda\" if torch.cuda.is_available() else \"cpu\"\n    torch.manual_seed(INIT_SEED)                     # \u0444\u0438\u043a\u0441\u0438\u0440\u043e\u0432\u0430\u043d\u043d\u044b\u0439 \u0438\u043d\u0438\u0442 \u2014 \u041d\u0415 args.seed\n    root = os.path.dirname(os.path.abspath(__file__))\n    os.makedirs(f\"{root}/{args.outdir}\", exist_ok=True)\n\n    tr = hnp.load(f\"{root}/{args.data}/train.npy\").astype(hnp.int64)\n    va = hnp.load(f\"{root}/{args.data}/val.npy\").astype(hnp.int64)\n    V = json.load(open(f\"{root}/{args.data}/meta.json\"))[\"vocab\"]\n    rng = hnp.random.default_rng(args.seed)\n    rng_neg = hnp.random.default_rng(args.seed * 1000003 + 17) if args.negrng else rng\n\n    model = LeanCore(V).to(dev)\n    import zlib as _zlib\n    np_paired_init(model, hnp, _zlib)   # \u0438\u043d\u0438\u0442 \u0431\u0438\u0442\u043e\u0432\u043e = numpy-\u0442\u0440\u0435\u043d\u0435\u0440 (\u0438\u043d\u0430\u0447\u0435 \u0433\u0435\u0439\u0442 \u043c\u0435\u0440\u044f\u0435\u0442 \u0448\u0443\u043c \u0438\u043d\u0438\u0442\u0430)\n    nparams = sum(p.numel() for p in model.parameters())\n    mu, ad = muon_split(model)\n    mbuf = {n: torch.zeros_like(p) for n, p in mu}\n    am = {n: torch.zeros_like(p) for n, p in ad}\n    av = {n: torch.zeros_like(p) for n, p in ad}\n    at = 0\n    print(f\"[{args.tag}] torch nano-ema params={nparams:,} dev={dev} muon={len(mu)} adam={len(ad)}\", flush=True)\n\n    ssq = None\n    if args.ssk > 0:\n        cnt = hnp.bincount(tr, minlength=V).astype(hnp.float64) + 1.0\n        ssq = cnt ** args.ssalpha\n        ssq = ssq / ssq.sum()\n\n    def batch(ids, rr):\n        st = rr.integers(0, len(ids) - args.ctx - 1, size=args.batch)\n        x = hnp.stack([ids[s:s + args.ctx] for s in st])\n        y = hnp.stack([ids[s + 1:s + args.ctx + 1] for s in st])\n        return torch.from_numpy(x).to(dev), torch.from_numpy(y).to(dev)\n\n    @torch.no_grad()\n    def vloss(iters=6):\n        model.eval()\n        rr = hnp.random.default_rng(1234); tot = 0.0\n        for _ in range(iters):\n            x, y = batch(va, rr)\n            lg = model.logits(model(x))\n            tot += F.cross_entropy(lg.reshape(-1, V), y.reshape(-1)).item()\n        model.train()\n        return tot / iters\n\n    def trunk_n0():\n        tot = 0.0\n        for n, p in mu:\n            v = p.detach().double().cpu()\n            tot += float((v * v).sum())\n        return tot ** 0.5\n\n    N0 = trunk_n0() if args.trunknorm else None\n    logf = open(f\"{root}/{args.outdir}/run_{args.tag}.jsonl\", \"w\")\n    t0 = time.time(); toks = 0; rec = {}\n    for step in range(args.steps):\n        lr = args.lr * min((step + 1) / args.warmup,\n              0.1 + 0.45 * (1 + math.cos(math.pi * max(0, step - args.warmup) / max(1, args.steps - args.warmup))))\n        x, y = batch(tr, rng)\n        h = model(x)\n        use_ss = ssq is not None and step < args.steps * (1.0 - args.ssfull)\n        if use_ss:\n            neg = rng_neg.choice(V, size=args.ssk, replace=True, p=ssq)\n            yh = y.cpu().numpy()\n            cand = hnp.union1d(hnp.unique(yh), neg).astype(hnp.int64)\n            logQ = hnp.log1p(-hnp.power(1.0 - ssq[cand], args.ssk))\n            logQ[hnp.isin(cand, yh)] = 0.0\n            lbl = hnp.searchsorted(cand, yh)\n            cand_t = torch.from_numpy(cand).to(dev)\n            logQ_t = torch.from_numpy(logQ).to(dev).to(f32)\n            lbl_t = torch.from_numpy(lbl).to(dev)\n            lg = h @ model.E[cand_t].t() - logQ_t[None, None, :]\n            loss = F.cross_entropy(lg.reshape(-1, lg.shape[-1]), lbl_t.reshape(-1))\n        else:\n            lg = model.logits(h)\n            loss = F.cross_entropy(lg.reshape(-1, V), y.reshape(-1))\n        model.zero_grad(set_to_none=True)\n        loss.backward()\n        with torch.no_grad():\n            at += 1\n            for n, p in mu:\n                g = p.grad\n                mbuf[n].lerp_(g, 1 - 0.95)\n                u = g.lerp(mbuf[n], 0.95)          # nesterov, \u043a\u0430\u043a nano_lc\n                O = ns5(u)\n                scale = max(1.0, O.shape[0] / O.shape[1]) ** 0.5\n                p.mul_(1 - args.wd * lr)\n                p.add_(O, alpha=-args.mulr * scale)\n            for n, p in ad:\n                g = p.grad\n                am[n].lerp_(g, 0.1)\n                av[n].mul_(0.95).addcmul_(g, g, value=0.05)\n                mh = am[n] / (1 - 0.9 ** at); vh = av[n] / (1 - 0.95 ** at)\n                p.mul_(1 - args.wd * lr)\n                p.addcdiv_(mh, vh.sqrt().add_(1e-8), value=-lr)\n        toks += x.numel()\n        if step % args.eval_every == 0 or step == args.steps - 1:\n            vl = vloss(); el = time.time() - t0\n            rec = dict(step=step, train_loss=round(float(loss.item()), 4), val_loss=round(vl, 4),\n                       val_ppl=round(float(math.exp(vl)), 2), wall=round(el, 1),\n                       tok_s=round(toks / el, 1), tokens=toks)\n            if N0 is not None:\n                rec[\"wn\"] = round(trunk_ratio(model, N0), 3)\n            print(json.dumps(rec), flush=True); logf.write(json.dumps(rec) + \"\\n\"); logf.flush()\n    if args.saveckpt:\n        m = {k: v.detach().cpu().numpy() for k, v in model.state_dict().items()}\n        hnp.savez(f\"{root}/{args.outdir}/ckpt_{args.tag}.npz\", **m)\n    print(\"SUMMARY \" + json.dumps(dict(tag=args.tag, kind=\"ema\", params=nparams,\n        tokens=toks, wall_s=round(time.time() - t0, 1), tok_s=round(toks / (time.time() - t0), 1),\n        final_val_loss=rec.get(\"val_loss\"), final_val_ppl=rec.get(\"val_ppl\"))), flush=True)\n\n\nif __name__ == \"__main__\":\n    main()\n")
print('train_torch.py ok')

In [ ]:
open(f'{BASE}/evo.py','w').write("#!/usr/bin/env python3\n# -*- coding: utf-8 -*-\n\"\"\"evo.py \u2014 \u044d\u0432\u043e\u043b\u044e\u0446\u0438\u043e\u043d\u043d\u044b\u0439 \u0434\u0432\u0438\u0436\u043e\u043a \u043f\u043e\u0438\u0441\u043a\u0430 \u043a\u043e\u043d\u0444\u0438\u0433\u043e\u0432 LeanCore (\u0441\u0442\u0430\u0434\u0438\u0438 A\u2192B\u2192C).\n\n\u0417\u0430\u0447\u0435\u043c (\u0437\u0430\u043c\u0435\u043d\u0430 \u00ab\u043f\u0440\u043e\u0441\u0442\u043e \u043f\u0435\u0440\u0435\u0431\u043e\u0440\u0443\u00bb): ASHA-\u0431\u0440\u0435\u043a\u0435\u0442 \u0432\u044b\u0431\u0438\u0440\u0430\u0435\u0442 \u043e\u0434\u0438\u043d \u0440\u0430\u0437 \u0438\u0437 \u0444\u0438\u043a\u0441\u0438\u0440\u043e\u0432\u0430\u043d\u043d\u043e\u0433\u043e\n\u043e\u0431\u043b\u0430\u043a\u0430 \u0438 \u043d\u0438\u0447\u0435\u043c\u0443 \u043d\u0435 \u0443\u0447\u0438\u0442\u0441\u044f. \u0417\u0434\u0435\u0441\u044c \u043f\u043e\u043f\u0443\u043b\u044f\u0446\u0438\u044f \u0420\u0410\u0417\u0412\u0418\u0412\u0410\u0415\u0422\u0421\u042f: \u0441\u0435\u043b\u0435\u043a\u0446\u0438\u044f \u2192 \u043a\u0440\u043e\u0441\u0441\u043e\u0432\u0435\u0440 \u2192\n\u043c\u0443\u0442\u0430\u0446\u0438\u0438 \u0441 \u0430\u0434\u0430\u043f\u0442\u0438\u0432\u043d\u044b\u043c \u0448\u0430\u0433\u043e\u043c (\u043f\u0440\u0430\u0432\u0438\u043b\u043e 1/5), \u0437\u0430\u043b \u0441\u043f\u0440\u0430\u0432\u0435\u0434\u043b\u0438\u0432\u043e\u0441\u0442\u0438 (HOF) \u043f\u043e \u0432\u0441\u0435\u0439 \u0438\u0441\u0442\u043e\u0440\u0438\u0438.\n\n\u041c\u0435\u0442\u043e\u0434\u043e\u043b\u043e\u0433\u0438\u044f \u043c\u0430\u0442\u0435\u0440\u0438\u043d\u0441\u043a\u043e\u0433\u043e \u043f\u0440\u043e\u0435\u043a\u0442\u0430 \u0441\u043e\u0445\u0440\u0430\u043d\u0435\u043d\u0430:\n  - \u043f\u0430\u0440\u043d\u044b\u0439 \u0434\u0435\u0442\u0435\u0440\u043c\u0438\u043d\u0438\u0440\u043e\u0432\u0430\u043d\u043d\u044b\u0439 \u0438\u043d\u0438\u0442 (crc32-\u0438\u043c\u0435\u043d\u0430 \u0443 numpy-\u0444\u043e\u0440\u043a\u0430; train_torch \u0441 v3 \u2014\n    \u0431\u0438\u0442\u043e\u0432\u043e \u0442\u043e\u0442 \u0436\u0435 \u0438\u043d\u0438\u0442), negrng-\u0440\u0430\u0437\u0434\u0435\u043b\u0435\u043d\u0438\u0435, \u043f\u0430\u0440\u043d\u044b\u0435 \u0441\u0438\u0434\u044b;\n  - CTRL (\u0440\u0435\u0446\u0435\u043f\u0442 \u0447\u0435\u043c\u043f\u0438\u043e\u043d\u0430) \u0432 \u043a\u0430\u0436\u0434\u043e\u043c \u043f\u043e\u043a\u043e\u043b\u0435\u043d\u0438\u0438 \u043a\u0430\u043a \u044f\u043a\u043e\u0440\u044c; \u0444\u0438\u0442\u043d\u0435\u0441 = \u0394PPL \u043a CTRL\n    \u042d\u0422\u041e\u0413\u041e \u0416\u0415 \u043f\u043e\u043a\u043e\u043b\u0435\u043d\u0438\u044f (\u0441\u043d\u0438\u043c\u0430\u0435\u0442 \u0434\u0440\u0435\u0439\u0444 \u0431\u0430\u0442\u0447\u0430/\u0434\u0430\u043d\u043d\u044b\u0445 \u043c\u0435\u0436\u0434\u0443 \u043f\u043e\u043a\u043e\u043b\u0435\u043d\u0438\u044f\u043c\u0438);\n  - \u0430\u0432\u0442\u043e-DQ \u00ab\u043c\u0451\u0440\u0442\u0432\u044b\u0445\u00bb (wn<3 \u043f\u043e\u0441\u043b\u0435 \u0448\u0430\u0433\u0430 100), \u0444\u0438\u043d\u0430\u043b 1500 \u00d7 \u0441\u0438\u0434\u044b [1,42,7,99];\n  - \u0432\u043e\u0437\u043e\u0431\u043d\u043e\u0432\u043b\u044f\u0435\u043c\u043e\u0441\u0442\u044c: evo_state.json + \u043b\u043e\u043a\u0438, \u043f\u0435\u0440\u0435\u0436\u0438\u0432\u0430\u0435\u0442 \u0442\u0430\u0439\u043c\u0430\u0443\u0442 \u0441\u0435\u0441\u0441\u0438\u0438 Kaggle.\n\n\u0421\u0442\u0430\u0434\u0438\u0438: A qual (16 \u043a\u043e\u043d\u0444\u0438\u0433\u043e\u0432: CTRL+15 LHS @60, \u0441\u0438\u0434 1 \u2014 \u0434\u0435\u0448\u0451\u0432\u044b\u0439 \u043e\u0442\u0441\u0435\u0432 \u0434\u043d\u0430) \u2192\nB evo (\u043f\u043e\u043f\u0443\u043b\u044f\u0446\u0438\u044f 8, \u043f\u043e\u043a\u043e\u043b\u0435\u043d\u0438\u0439 3 @200, \u0441\u0438\u0434\u044b [1,42]) \u2192 C final (HOF-3 + CTRL\n@1500 \u00d7 4 \u0441\u0438\u0434\u0430). \u041f\u0440\u043e\u0433\u043d\u043e\u0437\u044b \u2014 PREDICTIONS \u043d\u0438\u0436\u0435, \u0437\u0430\u0444\u0438\u043a\u0441\u0438\u0440\u043e\u0432\u0430\u043d\u044b \u0414\u041e \u0437\u0430\u043f\u0443\u0441\u043a\u0430.\n\n\u0420\u0435\u0436\u0438\u043c\u044b CLI: init | worker | status | summary. Env EVO_QUICK=1 \u2014 \u0442\u0440\u0443\u0431\u043e\u043f\u0440\u043e\u0432\u043e\u0434\u043d\u044b\u0439\n\u0434\u044b\u043c (qual 4 @20, 2 \u043f\u043e\u043a\u043e\u043b\u0435\u043d\u0438\u044f \u043f\u043e 4, \u0444\u0438\u043d\u0430\u043b @30).\n\"\"\"\nimport os, sys, json, math, time, glob, shutil, argparse, subprocess\nimport numpy as hnp\n\nQUICK = os.environ.get(\"EVO_QUICK\", \"0\") == \"1\"\nHERE = os.path.dirname(os.path.abspath(__file__))\n\n# \u0442\u043e\u0447\u043a\u0430 \u043e\u0442\u0441\u0447\u0451\u0442\u0430 \u2014 \u0440\u0435\u0446\u0435\u043f\u0442 \u0447\u0435\u043c\u043f\u0438\u043e\u043d\u0430 (\u043a\u0430\u043a \u0432 bracket.py)\nCTRL = dict(name=\"CTRL\", mulr=0.02, ssfull=0.12, ssk=512, ssalpha=1.0, lr=6e-4, origin=\"ctrl\")\n\nBOUNDS = dict(mulr=(0.010, 0.040), ssfull=(0.0, 0.30), ssk=(256, 768), ssalpha=(0.5, 1.0), lr=(4e-4, 9e-4))\nSSK_STEPS = [256, 320, 384, 512, 768]\n\nif QUICK:\n    QUAL_N, QUAL_STEPS, QUAL_SEEDS = 3, 20, [1]\n    POP, GENS, EVO_STEPS, EVO_SEEDS = 4, 2, 20, [1]\n    FINAL_TOP, FINAL_STEPS, FINAL_SEEDS = 2, 30, [1]\nelse:\n    QUAL_N, QUAL_STEPS, QUAL_SEEDS = 15, 60, [1]\n    POP, GENS, EVO_STEPS, EVO_SEEDS = 8, 3, 200, [1, 42]\n    FINAL_TOP, FINAL_STEPS, FINAL_SEEDS = 3, 1500, [1, 42, 7, 99]\n\nMASTER_SEED = 0xE90\n\nPREDICTIONS = \"\"\"# \u041f\u0420\u0415\u0414\u0421\u041a\u0410\u0417\u0410\u041d\u0418\u042f EVO-\u041a\u041e\u041d\u0422\u0423\u0420\u0410 (\u0437\u0430\u0444\u0438\u043a\u0441\u0438\u0440\u043e\u0432\u0430\u043d\u044b \u0414\u041e \u0437\u0430\u043f\u0443\u0441\u043a\u0430, \u0441\u043d\u044f\u0442\u044c \u043f\u043e\u0441\u043b\u0435 SUMMARY)\nE1. \u0422\u0435\u0445\u043d\u0438\u0447\u0435\u0441\u043a\u0438\u0439: \u0438\u0441\u0442\u043e\u0440\u0438\u044f \u043f\u043e\u043a\u043e\u043b\u0435\u043d\u0438\u0439 \u043f\u0440\u0438 \u0442\u043e\u043c \u0436\u0435 MASTER_SEED \u0432\u043e\u0441\u043f\u0440\u043e\u0438\u0437\u0432\u043e\u0434\u0438\u043c\u0430 \u0431\u0438\u0442\u043e\u0432\u043e\n    (\u0442\u0440\u0430\u0441\u0441\u0430 \u043a\u043e\u043d\u0444\u0438\u0433\u043e\u0432, \u043d\u0435 \u0432\u0435\u0441\u043e\u0432) \u0432 \u0434\u0432\u0443\u0445 \u043d\u0435\u0437\u0430\u0432\u0438\u0441\u0438\u043c\u044b\u0445 \u0441\u0435\u0441\u0441\u0438\u044f\u0445 [\u043a\u043e\u043d\u0442\u0440\u043e\u043b\u044c \u043c\u0435\u0445\u0430\u043d\u0438\u0437\u043c\u0430].\nE2. \u041f\u043e\u0438\u0441\u043a\u043e\u0432\u044b\u0439: \u043a \u043f\u043e\u043a\u043e\u043b\u0435\u043d\u0438\u044e 3 \u043c\u0435\u0434\u0438\u0430\u043d\u0430 ssalpha \u043f\u043e\u043f\u0443\u043b\u044f\u0446\u0438\u0438 \u2264 1.0 \u2014 \u043c\u0435\u0445\u0430\u043d\u0438\u0437\u043c \u0421\u0410\u041c \u0441\u043c\u0435\u0441\u0442\u0438\u0442\u0441\u044f\n    \u0432\u043d\u0438\u0437 \u043f\u043e \u03b1 (\u03b1=0.75 \u0434\u0430\u043b \u22121.30% @1500 \u0432 \u0447\u0438\u0441\u0442\u043e\u0439 \u043f\u0430\u0440\u0435 [\u0438\u0437\u043c\u0435\u0440\u0435\u043d\u043e \u0434\u043e\u043c\u0430]; \u043f\u043e\u0434\u0441\u043a\u0430\u0437\u043a\u0438 \u043d\u0435\u0442).\nE3. \u0418\u0442\u043e\u0433\u043e\u0432\u044b\u0439: \u043b\u0443\u0447\u0448\u0438\u0439 HOF-\u043a\u043e\u043d\u0444\u0438\u0433 \u043d\u0430 \u0444\u0438\u043d\u0430\u043b\u0435 (4 \u0441\u0438\u0434\u0430) \u0431\u0443\u0434\u0435\u0442 \u041d\u0415 \u0445\u0443\u0436\u0435 CTRL \u0431\u043e\u043b\u0435\u0435 \u0447\u0435\u043c \u043d\u0430\n    +0.5% \u0438, \u043e\u0436\u0438\u0434\u0430\u0435\u043c\u043e, \u22120.5\u2026\u22122.5% \u043a CTRL (\u044f\u043a\u043e\u0440\u044c \u2014 \u03b1-\u043f\u0430\u0440\u0430). \u0415\u0441\u043b\u0438 \u0445\u0443\u0436\u0435 +0.5% \u2014 \u0441\u0438\u0433\u043d\u0430\u043b\n    @200 \u043d\u0438\u0436\u0435 \u043d\u0430\u0448\u0435\u0433\u043e \u0448\u0443\u043c\u0430, \u043a\u043e\u043d\u0442\u0443\u0440 \u0432 \u0442\u0430\u043a\u043e\u043c \u0432\u0438\u0434\u0435 \u043f\u0430\u0440\u043a\u0438\u043d\u0433\u0443\u0435\u0442\u0441\u044f [\u043e\u0442\u0440\u0438\u0446\u0430\u0442\u0435\u043b\u044c\u043d\u0430\u044f \u0432\u0435\u0442\u043a\u0430].\nE4. \u0421\u0442\u0440\u0443\u043a\u0442\u0443\u0440\u043d\u044b\u0439: \u0443\u0441\u043f\u0435\u0448\u043d\u043e\u0441\u0442\u044c \u043a\u0440\u043e\u0441\u0441\u043e\u0432\u0435\u0440\u043e\u0432 \u2264 \u0443\u0441\u043f\u0435\u0448\u043d\u043e\u0441\u0442\u0438 \u043c\u0443\u0442\u0430\u0446\u0438\u0439 (\u043f\u0440\u043e\u0441\u0442\u0440\u0430\u043d\u0441\u0442\u0432\u043e \u043f\u043e\u0447\u0442\u0438\n    \u0441\u0435\u043f\u0430\u0440\u0430\u0431\u0435\u043b\u044c\u043d\u043e\u0435 \u2014 \u043d\u0430\u0448 \u0430\u0440\u0445\u0438\u0432 \u044d\u0442\u043e \u043f\u043e\u0434\u0434\u0435\u0440\u0436\u0438\u0432\u0430\u0435\u0442) [\u043f\u0440\u043e\u0432\u0435\u0440\u0438\u0442\u0441\u044f \u043f\u043e \u0442\u0440\u0430\u0441\u0441\u0435].\nE5. \u042f\u043a\u043e\u0440\u043d\u044b\u0439: CTRL \u043d\u0435 \u0432\u044b\u0431\u044b\u0432\u0430\u0435\u0442 \u0438\u0437 \u0442\u043e\u043f-3 \u043d\u0438 \u0432 \u043e\u0434\u043d\u043e\u043c \u043f\u043e\u043a\u043e\u043b\u0435\u043d\u0438\u0438 \u0431\u043e\u043b\u044c\u0448\u0435 \u0447\u0435\u043c \u043f\u043e \u0432\u0438\u043d\u0435 \u22641 \u043c\u0443\u0442\u0430\u043d\u0442\u0430\n    (\u043f\u043e\u043b \u043e\u0434\u043d\u043e-\u043f\u0430\u0440\u043d\u043e\u0433\u043e \u0448\u0443\u043c\u0430 \u0443 \u043d\u0430\u0441 \u00b11.5\u20132.5% \u2014 \u0441\u0438\u0441\u0442\u0435\u043c\u0430\u0442\u0438\u0447\u0435\u0441\u043a\u0438\u0439 \u043a\u0430\u0441\u043a\u0430\u0434 \u0432\u044b\u0431\u0438\u0432\u0430\u043d\u0438\u0439 \u043e\u0437\u043d\u0430\u0447\u0430\u043b \u0431\u044b\n    \u043f\u0435\u0440\u0435\u043e\u0431\u0443\u0447\u0435\u043d\u0438\u0435 \u043c\u0435\u0445\u0430\u043d\u0438\u0437\u043c\u0430 \u0432 \u0448\u0443\u043c).\n\"\"\"\n\n\n# ----------------------------------------------------------------- \u0433\u0435\u043d\u043e\u043c \u0438 \u043e\u043f\u0435\u0440\u0430\u0442\u043e\u0440\u044b\ndef _ckey(c):\n    return (c[\"mulr\"], c[\"ssfull\"], c[\"ssk\"], c[\"ssalpha\"], c[\"lr\"])\n\n\ndef _add_unique(pop, cand):\n    if all(_ckey(cand) != _ckey(c) for c in pop):\n        pop.append(cand)\n        return True\n    return False\ndef to_gene(c):\n    \"\"\"\u041a\u043e\u043d\u0444\u0438\u0433 \u2192 \u0442\u043e\u0447\u043a\u0430 \u0432 \u0433\u0435\u043d\u0435-\u043f\u0440\u043e\u0441\u0442\u0440\u0430\u043d\u0441\u0442\u0432\u0435 (\u043b\u043e\u0433-\u043a\u043e\u043e\u0440\u0434\u0438\u043d\u0430\u0442\u044b \u0442\u0430\u043c, \u0433\u0434\u0435 \u043c\u0430\u0441\u0448\u0442\u0430\u0431 \u043c\u0443\u043b\u044c\u0442\u0438\u043f\u043b\u0438\u043a\u0430\u0442\u0438\u0432\u0435\u043d).\"\"\"\n    return dict(l_mulr=math.log(c[\"mulr\"]), l_lr=math.log(c[\"lr\"]),\n                l_ssfull=math.log(c[\"ssfull\"] + 0.02), a= c[\"ssalpha\"], k=SSK_STEPS.index(\n                    min(SSK_STEPS, key=lambda s: abs(s - c[\"ssk\"]))))\n\n\ndef from_gene(g):\n    def cl(x, lo, hi):\n        return max(lo, min(hi, x))\n    return dict(mulr=round(cl(math.exp(g[\"l_mulr\"]), *BOUNDS[\"mulr\"]), 4),\n                ssfull=round(cl(math.exp(g[\"l_ssfull\"]) - 0.02, *BOUNDS[\"ssfull\"]), 3),\n                ssk=SSK_STEPS[int(round(cl(g[\"k\"], 0, len(SSK_STEPS) - 1)))],\n                ssalpha=round(cl(g[\"a\"], *BOUNDS[\"ssalpha\"]), 3),\n                lr=round(cl(math.exp(g[\"l_lr\"]), *BOUNDS[\"lr\"]), 7))\n\n\ndef mutate(cfg, rng, sigma):\n    g = to_gene(cfg)\n    g[\"l_mulr\"] += rng.normal(0, sigma)\n    g[\"l_lr\"] += rng.normal(0, sigma * 0.8)\n    g[\"l_ssfull\"] += rng.normal(0, sigma)\n    g[\"a\"] += rng.normal(0, sigma * 0.35)\n    if rng.random() < min(0.75, sigma * 3.0):       # \u0441\u0442\u0443\u043f\u0435\u043d\u0447\u0430\u0442\u0430\u044f \u043a\u043e\u043e\u0440\u0434\u0438\u043d\u0430\u0442\u0430\n        g[\"k\"] += int(rng.choice([-1, 1]))\n    out = from_gene(g)\n    out.update(name=\"\", origin=f\"mut({cfg['name']})\")\n    return out\n\n\ndef crossover(ca, cb, rng):\n    out = {}\n    for key in (\"mulr\", \"ssfull\", \"ssk\", \"ssalpha\", \"lr\"):\n        out[key] = ca[key] if rng.random() < 0.5 else cb[key]\n    out.update(name=\"\", origin=f\"x({ca['name']}\u00d7{cb['name']})\")\n    return out\n\n\n# ----------------------------------------------------------------- state / storage\ndef wp(workdir, *p):\n    return os.path.join(workdir, *p)\n\n\ndef load(workdir):\n    p = wp(workdir, \"evo_state.json\")\n    if os.path.exists(p):\n        return json.load(open(p))\n    return {\"stage\": \"qual\", \"gen\": 0, \"sigma\": 0.18, \"jobs\": {}, \"pop\": [],\n            \"history\": [], \"hof\": {}, \"qual_cfgs\": [], \"created\": int(time.time())}\n\n\ndef save(workdir, st):\n    tmp = wp(workdir, \"evo_state.json.tmp\")\n    json.dump(st, open(tmp, \"w\"), ensure_ascii=False, indent=1)\n    os.replace(tmp, wp(workdir, \"evo_state.json\"))\n\n\ndef jkey(stage, gen, name, seed):\n    return f\"{stage}{gen}_{name}_s{seed}\"\n\n\ndef cfg_cli(engine, cfg, steps, eval_every, seed, tag, data, saveckpt):\n    if engine == \"torch\":\n        return [sys.executable, \"train_torch.py\", \"--steps\", str(steps), \"--eval_every\", str(eval_every),\n                \"--lr\", str(cfg[\"lr\"]), \"--mulr\", str(cfg[\"mulr\"]), \"--ssk\", str(cfg[\"ssk\"]),\n                \"--ssfull\", str(cfg[\"ssfull\"]), \"--ssalpha\", str(cfg[\"ssalpha\"]), \"--seed\", str(seed),\n                \"--negrng\", \"1\", \"--trunknorm\", \"1\", \"--tag\", tag, \"--data\", data,\n                \"--saveckpt\", str(saveckpt)]\n    return [sys.executable, \"nano_lc_kg.py\", \"--kind\", \"ema\", \"--opt\", \"muon\", \"--steps\", str(steps),\n            \"--eval_every\", str(eval_every), \"--lr\", str(cfg[\"lr\"]), \"--mulr\", str(cfg[\"mulr\"]),\n            \"--ssk\", str(cfg[\"ssk\"]), \"--ssfull\", str(cfg[\"ssfull\"]), \"--ssalpha\", str(cfg[\"ssalpha\"]),\n            \"--seed\", str(seed), \"--negrng\", \"1\", \"--trunknorm\", \"1\", \"--tag\", tag, \"--data\", data]\n\n\ndef lhs_cfgs(n):\n    \"\"\"CTRL + n LHS-\u0442\u043e\u0447\u0435\u043a \u043f\u0440\u043e\u0441\u0442\u0440\u0430\u043d\u0441\u0442\u0432\u0430 (\u0442\u043e\u0442 \u0436\u0435 \u0433\u0435\u043d\u0435\u0440\u0430\u0442\u043e\u0440, \u0447\u0442\u043e \u0432 bracket, \u2014 \u0441\u043e\u043f\u043e\u0441\u0442\u0430\u0432\u0438\u043c\u043e\u0441\u0442\u044c).\"\"\"\n    rng = hnp.random.default_rng(20240830)\n    pts = hnp.zeros((n, 5))\n    for j in range(5):\n        perm = rng.permutation(n)\n        pts[:, j] = (perm + rng.random(n)) / n\n    out = [dict(CTRL)]\n    for i in range(n):\n        q = dict(name=f\"Q{i:02d}\",\n                 mulr=round(0.010 * 4.0 ** pts[i, 0], 4),\n                 ssfull=round(0.30 * pts[i, 1], 3),\n                 ssk=min(int(round(256 * 3.0 ** pts[i, 2] / 32) * 32), 768),\n                 ssalpha=round(0.5 + 0.5 * pts[i, 3], 3),\n                 lr=round(4e-4 * (9e-4 / 4e-4) ** pts[i, 4], 7), origin=\"lhs\")\n        out.append(q)\n    return out\n\n\ndef queue_stage_jobs(st, stage, gen, cfgs, steps, seeds):\n    ee = max(10, steps // 6)\n    for cfg in cfgs:\n        for seed in seeds:\n            k = jkey(stage, gen, cfg[\"name\"], seed)\n            if k not in st[\"jobs\"]:\n                st[\"jobs\"][k] = dict(stage=stage, gen=gen, cfg=dict(cfg), steps=steps,\n                                     eval_every=ee, seed=seed, status=\"pending\",\n                                     tag=f\"ev_{k}\")\n\n\ndef parse_result(job):\n    path = os.path.join(HERE, \"results\", f\"run_{job['tag']}.jsonl\")\n    if not os.path.exists(path):\n        return \"dq\", None, 0.0, \"\u043d\u0435\u0442 jsonl\"\n    rows = [json.loads(l) for l in open(path) if l.strip().startswith(\"{\")]\n    if not rows:\n        return \"dq\", None, 0.0, \"\u043f\u0443\u0441\u0442\u043e\u0439 jsonl\"\n    last = rows[-1]\n    ppl = last.get(\"val_ppl\")\n    if ppl is None or not math.isfinite(ppl):\n        return \"dq\", None, 0.0, \"NaN\"\n    for r in rows:\n        if r[\"step\"] >= 100 and \"wn\" in r and r[\"wn\"] < 3.0:\n            return \"dq\", float(ppl), 0.0, f\"\u043c\u0435\u0440\u0442\u0432: wn={r['wn']}@{r['step']}\"\n    slope = ((rows[0][\"val_ppl\"] - ppl) / max(1, last[\"step\"] - rows[0][\"step\"])) if len(rows) > 1 else 0.0\n    return \"done\", float(ppl), float(slope), \"\"\n\n\n# ----------------------------------------------------------------- \u0440\u0435\u0448\u0435\u043d\u0438\u044f \u0441\u0442\u0430\u0434\u0438\u0439\ndef gen_jobs(st, stage, gen):\n    return [j for j in st[\"jobs\"].values() if j[\"stage\"] == stage and j[\"gen\"] == gen]\n\n\ndef stage_done(js):\n    return js and all(j[\"status\"] in (\"done\", \"dq\") for j in js)\n\n\ndef score_by_cfg(js):\n    by = {}\n    for j in js:\n        if j[\"status\"] == \"done\":\n            by.setdefault(j[\"cfg\"][\"name\"], []).append(j[\"val_ppl\"])\n    return {k: float(hnp.mean(v)) for k, v in by.items()}\n\n\ndef advance(workdir, st):\n    \"\"\"\u041b\u043e\u043a \u043d\u0443\u0436\u0435\u043d \u0441\u043d\u0430\u0440\u0443\u0436\u0438. \u0420\u0435\u0448\u0430\u0435\u0442 \u0437\u0430\u0432\u0435\u0440\u0448\u0451\u043d\u043d\u0443\u044e \u0441\u0442\u0430\u0434\u0438\u044e \u0438 \u0440\u0430\u0441\u043a\u043b\u0430\u0434\u044b\u0432\u0430\u0435\u0442 \u0441\u043b\u0435\u0434\u0443\u044e\u0449\u0443\u044e.\"\"\"\n    stage, gen = st[\"stage\"], st[\"gen\"]\n    js = gen_jobs(st, stage, gen)\n    if stage == \"qual\":\n        sc = score_by_cfg(js)\n        cjob = {j[\"cfg\"][\"name\"]: j[\"cfg\"] for j in js}\n        ranked = sorted(sc.items(), key=lambda t: t[1])\n        st[\"history\"].append(dict(stage=\"qual\", gen=0,\n                                  table=[dict(cfg=k, ppl=round(v, 2)) for k, v in ranked]))\n        top2 = [cjob[k] for k, _ in ranked if k != \"CTRL\"][:2]\n        rng = hnp.random.default_rng(hnp.random.SeedSequence([MASTER_SEED, 0]))\n        pop = [dict(CTRL)] + top2\n        guard = 0\n        while len(pop) < POP - 3 and guard < 24:\n            guard += 1\n            _add_unique(pop, mutate(rng.choice(top2), rng, st[\"sigma\"]))\n        for cand in [crossover(top2[0], dict(CTRL), rng), crossover(top2[-1], dict(CTRL), rng),\n                     mutate(dict(CTRL), rng, st[\"sigma\"])]:\n            _add_unique(pop, cand)\n        guard = 0\n        while len(pop) < POP and guard < 24:\n            guard += 1\n            _add_unique(pop, mutate(rng.choice(top2), rng, st[\"sigma\"]))\n        pop = pop[:POP]\n        for i, c in enumerate(pop):\n            if not c[\"name\"].startswith((\"CTRL\", \"Q\")) or sum(p[\"name\"] == c[\"name\"] for p in pop) > 1:\n                c[\"name\"] = f\"G1.{i}\"\n        st[\"pop\"] = [dict(c) for c in pop]\n        st[\"stage\"], st[\"gen\"] = \"evo\", 1\n        queue_stage_jobs(st, \"evo\", 1, pop, EVO_STEPS, EVO_SEEDS)\n        return f\"qual \u0440\u0435\u0448\u0451\u043d \u2192 \u043f\u043e\u043f\u0443\u043b\u044f\u0446\u0438\u044f G1: {[c['name'] for c in pop]}\"\n    if stage == \"evo\":\n        sc = score_by_cfg(js)\n        ctrl_ppl = sc.get(\"CTRL\")\n        genes = {j[\"cfg\"][\"name\"]: j[\"cfg\"] for j in js}\n        tbl = []\n        for name, ppl in sorted(sc.items(), key=lambda t: t[1]):\n            tbl.append(dict(cfg=name, ppl=round(ppl, 2),\n                            dctrl=round(ppl - ctrl_ppl, 2) if ctrl_ppl else None,\n                            origin=genes[name].get(\"origin\", \"\")))\n        ranked = [t[\"cfg\"] for t in tbl if t[\"cfg\"] != \"CTRL\"]\n        top3_names = ranked[:3]\n        elites = [genes[n] for n in ranked[:2]] + [dict(CTRL)]   # CTRL \u2014 \u0432\u0435\u0447\u043d\u044b\u0439 \u044f\u043a\u043e\u0440\u043d\u044b\u0439 \u044d\u043b\u0438\u0442\n        if ctrl_ppl:  # HOF: \u043b\u0443\u0447\u0448\u0438\u0439 dctrl \u043a\u043e\u043d\u0444\u0438\u0433\u0430 \u0437\u0430 \u0432\u0441\u044e \u0438\u0441\u0442\u043e\u0440\u0438\u044e\n            for name, ppl in sc.items():\n                if name == \"CTRL\":\n                    continue\n                fit = ctrl_ppl - ppl\n                if fit > st[\"hof\"].get(\"fit\", float(\"-inf\")):\n                    st[\"hof\"] = dict(cfg=dict(genes[name]), fit=round(fit, 2), gen=gen)\n        succ = sum(1 for j in js if j[\"status\"] == \"done\" and \"mut\" in j[\"cfg\"].get(\"origin\", \"\")\n                   and j[\"cfg\"][\"name\"] in top3_names)\n        nmut = max(1, sum(1 for j in js if \"mut\" in j[\"cfg\"].get(\"origin\", \"\")))\n        ps = succ / nmut\n        st[\"sigma\"] = round(min(0.5, max(0.06, st[\"sigma\"] * (1.22 if ps > 0.2 else 0.82))), 3)\n        st[\"history\"].append(dict(stage=\"evo\", gen=gen, table=tbl, sigma=st[\"sigma\"],\n                                  ps=round(ps, 2), top3=top3_names))\n        if gen >= GENS:\n            st[\"stage\"], st[\"gen\"] = \"final\", 0\n            finalists = [dict(CTRL)]\n            if st[\"hof\"] and _ckey(st[\"hof\"][\"cfg\"]) != _ckey(CTRL):\n                finalists.append(dict(st[\"hof\"][\"cfg\"]))\n            for e in elites:\n                if all(_ckey(e) != _ckey(f) for f in finalists):\n                    finalists.append(dict(e))\n                if len(finalists) >= FINAL_TOP + 1:\n                    break\n            queue_stage_jobs(st, \"final\", 0, finalists, FINAL_STEPS, FINAL_SEEDS)\n            return f\"evo \u0444\u0438\u043d\u0438\u0448 \u2192 \u0444\u0438\u043d\u0430\u043b: {[f['name'] for f in finalists]}, HOF={st['hof'] and st['hof']['cfg']['name']}\"\n        rng = hnp.random.default_rng(hnp.random.SeedSequence([MASTER_SEED, gen]))\n        pop = [dict(e) for e in elites]\n        guard = 0\n        while len(pop) < POP - 1 and guard < 24:\n            guard += 1\n            _add_unique(pop, mutate(elites[rng.integers(0, len(elites))], rng, st[\"sigma\"]))\n        if not _add_unique(pop, crossover(elites[0], elites[-1], rng)):\n            _add_unique(pop, mutate(elites[0], rng, st[\"sigma\"]))\n        for i, c in enumerate(pop):\n            if c[\"name\"] != \"CTRL\":\n                c[\"name\"] = f\"G{gen+1}.{i}\"\n        st[\"pop\"] = [dict(c) for c in pop]\n        st[\"gen\"] = gen + 1\n        queue_stage_jobs(st, \"evo\", gen + 1, pop, EVO_STEPS, EVO_SEEDS)\n        return f\"\u043f\u043e\u043a\u043e\u043b\u0435\u043d\u0438\u0435 {gen} \u0440\u0435\u0448\u0435\u043d\u043e (ps={ps:.2f}, \u03c3\u2192{st['sigma']}) \u2192 G{gen+1}\"\n    return \"final \u0437\u0430\u0432\u0435\u0440\u0448\u0451\u043d\"\n\n\n# ----------------------------------------------------------------- CLI\ndef cmd_init(args):\n    os.makedirs(wp(args.workdir, \"jobs\"), exist_ok=True)\n    for stale in glob.glob(wp(args.workdir, \"jobs\", \"*.lock\")):\n        shutil.rmtree(stale, ignore_errors=True)\n    if not os.path.exists(wp(args.workdir, \"evo_state.json\")):\n        for cand in glob.glob(\"/kaggle/input/*/evo_state.json\"):\n            shutil.copy(cand, wp(args.workdir, \"evo_state.json\"))\n            print(f\"[evo-init] state \u0432\u043e\u0441\u0441\u0442\u0430\u043d\u043e\u0432\u043b\u0435\u043d \u0438\u0437 {cand}\", flush=True)\n            break\n    st = load(args.workdir)\n    n_reset = 0\n    for j in st[\"jobs\"].values():\n        if j[\"status\"] == \"running\":\n            j[\"status\"] = \"pending\"; n_reset += 1\n    if n_reset:\n        print(f\"[evo-init] \u0437\u0430\u0432\u0438\u0441\u0448\u0438\u0435 running \u2192 pending: {n_reset}\", flush=True)\n    if not st[\"jobs\"] and not st[\"history\"]:\n        qual = lhs_cfgs(QUAL_N)\n        queue_stage_jobs(st, \"qual\", 0, qual, QUAL_STEPS, QUAL_SEEDS)\n        print(f\"[evo-init] qual: {len(qual)} \u043a\u043e\u043d\u0444\u0438\u0433\u043e\u0432 @{QUAL_STEPS}\", flush=True)\n    save(args.workdir, st)\n    open(wp(args.workdir, \"PREDICTIONS.md\"), \"w\").write(PREDICTIONS)\n\n\ndef cmd_worker(args):\n    env = dict(os.environ)\n    if args.backend:\n        env[\"LC_BACKEND\"] = args.backend\n    if args.cuda is not None:\n        env[\"CUDA_VISIBLE_DEVICES\"] = str(args.cuda)\n    while True:\n        st = load(args.workdir)\n        if st[\"stage\"] == \"final\" and stage_done(gen_jobs(st, \"final\", 0)):\n            print(f\"[w{args.id}] \u0444\u0438\u043d\u0430\u043b \u0437\u0430\u0432\u0435\u0440\u0448\u0451\u043d\", flush=True)\n            return\n        cur = gen_jobs(st, st[\"stage\"], st[\"gen\"])\n        if stage_done(cur) and st[\"stage\"] != \"final\":\n            lock = wp(args.workdir, \"jobs\", \"advance.lock\")\n            try:\n                os.mkdir(lock)\n                st = load(args.workdir)\n                cur = gen_jobs(st, st[\"stage\"], st[\"gen\"])\n                if stage_done(cur):\n                    msg = advance(args.workdir, st)\n                    save(args.workdir, st)\n                    print(f\"[w{args.id}] {msg}\", flush=True)\n            finally:\n                shutil.rmtree(lock, ignore_errors=True)\n        elif stage_done(cur) and st[\"stage\"] == \"final\":\n            return\n        claimed = None\n        for _attempt in range(10):          # \u0442\u0435\u0440\u043f\u0438\u043c: \u0434\u0440\u0443\u0433\u043e\u0439 \u0432\u043e\u0440\u043a\u0435\u0440 \u043c\u043e\u0436\u0435\u0442 \u0441\u0435\u0439\u0447\u0430\u0441 advance'\u043d\u0443\u0442\u044c\n            st = load(args.workdir)\n            for j in sorted((j for j in st[\"jobs\"].values() if j[\"status\"] == \"pending\"),\n                            key=lambda j: j[\"tag\"]):\n                lock = wp(args.workdir, \"jobs\", f\"{j['tag']}.lock\")\n                try:\n                    os.mkdir(lock)\n                    claimed = (j, lock); break\n                except FileExistsError:\n                    continue\n            if claimed or st[\"stage\"] == \"final\" or all(\n                    j[\"status\"] in (\"done\", \"dq\") for j in st[\"jobs\"].values()):\n                break\n            time.sleep(20)\n        if claimed is None:\n            print(f\"[w{args.id}] \u043d\u0435\u0442 \u0434\u043e\u0441\u0442\u0443\u043f\u043d\u044b\u0445 \u0434\u0436\u043e\u0431; \u0432\u044b\u0445\u043e\u0434\", flush=True)\n            return\n        job, lock = claimed\n        key = [k for k, v in st[\"jobs\"].items() if v[\"tag\"] == job[\"tag\"]][0]\n        st = load(args.workdir)\n        st[\"jobs\"][key][\"status\"] = \"running\"\n        save(args.workdir, st)\n        cmd = cfg_cli(args.engine, job[\"cfg\"], job[\"steps\"], job[\"eval_every\"], job[\"seed\"],\n                      job[\"tag\"], args.data, saveckpt=(1 if st_stage_is_final(job) else 0))\n        print(f\"[w{args.id}] \u0421\u0422\u0410\u0420\u0422 {key} (steps={job['steps']} seed={job['seed']} cfg={job['cfg']})\", flush=True)\n        t0 = time.time()\n        try:\n            proc = subprocess.run(cmd, cwd=HERE, env=env, timeout=args.job_timeout)\n            rc = proc.returncode\n        except subprocess.TimeoutExpired:\n            rc = -9\n        status, ppl, slope, dq = parse_result(job)\n        if rc != 0 and status == \"done\":\n            status, dq = \"dq\", f\"rc={rc}\"\n        st = load(args.workdir)\n        st[\"jobs\"][key].update(status=status, val_ppl=ppl, slope=slope, dq=dq,\n                               worker=args.id, elapsed=round(time.time() - t0, 1))\n        save(args.workdir, st)\n        shutil.rmtree(lock, ignore_errors=True)\n        print(f\"[w{args.id}] \u0424\u0418\u041d {key}: {status} ppl={ppl} {dq}\", flush=True)\n\n\ndef st_stage_is_final(job):\n    return job[\"stage\"] == \"final\"\n\n\ndef cmd_status(args):\n    st = load(args.workdir)\n    print(f\"stage={st['stage']} gen={st['gen']} \u03c3={st['sigma']} \u043f\u043e\u043f\u0443\u043b\u044f\u0446\u0438\u044f={len(st.get('pop') or [])}\")\n    for h in st[\"history\"]:\n        print(f\"  \u0438\u0441\u0442\u043e\u0440\u0438\u044f: {h['stage']}{h.get('gen')} \" +\n              (f\"\u03c3={h.get('sigma')} ps={h.get('ps')} \u0442\u043e\u043f={h.get('top3')}\" if h[\"stage\"] == \"evo\" else \"\"))\n    for stage in (\"qual\", \"evo\", \"final\"):\n        for g in (0, 1, 2, 3):\n            js = [j for j in st[\"jobs\"].values() if j[\"stage\"] == stage and j[\"gen\"] == g]\n            if js:\n                done = sum(1 for j in js if j[\"status\"] in (\"done\", \"dq\"))\n                print(f\"  {stage}{g}: {done}/{len(js)}\")\n\n\ndef cmd_summary(args):\n    st = load(args.workdir)\n    L = [\"# EVO-\u0422\u0423\u0420\u041d\u0418\u0420 LeanCore \u2014 \u0438\u0442\u043e\u0433\u0438\", \"\",\n         f\"\u043f\u043e\u043a\u043e\u043b\u0435\u043d\u0438\u0439 \u0441\u044b\u0433\u0440\u0430\u043d\u043e: {st['gen'] if st['stage'] == 'evo' else st['gen']}, \u0444\u0438\u043d\u0430\u043b\u044c\u043d\u044b\u0439 \u03c3={st['sigma']}\",\n         f\"HOF: {json.dumps(st['hof'], ensure_ascii=False)}\", \"\"]\n    for h in st[\"history\"]:\n        if h[\"stage\"] == \"qual\":\n            L += [f\"## \u041a\u0432\u0430\u043b\u0438\u0444\u0438\u043a\u0430\u0446\u0438\u044f @{QUAL_STEPS}\u0448 (\u0441\u0438\u0434 {QUAL_SEEDS})\", \"| cfg | PPL |\", \"|---|---|\"]\n            L += [f\"| {t['cfg']} | {t['ppl']} |\" for t in h[\"table\"]]\n        else:\n            L += [f\"## \u041f\u043e\u043a\u043e\u043b\u0435\u043d\u0438\u0435 {h['gen']} @{EVO_STEPS}\u0448 (\u0441\u0438\u0434\u044b {EVO_SEEDS}) \u2014 \u03c3={h['sigma']}, ps={h['ps']}\",\n                  \"| cfg | PPL | \u0394PPL \u043a CTRL | \u043f\u0440\u043e\u0438\u0441\u0445\u043e\u0436\u0434\u0435\u043d\u0438\u0435 |\", \"|---|---|---|---|\"]\n            L += [f\"| {t['cfg']} | {t['ppl']} | {t['dctrl']} | {t['origin']} |\" for t in h[\"table\"]]\n            L += [f\"\u0442\u043e\u043f-3: {h['top3']}\"]\n        L.append(\"\")\n    fin = [j for j in st[\"jobs\"].values() if j[\"stage\"] == \"final\"]\n    if fin:\n        L += [\"## \u0424\u0438\u043d\u0430\u043b @1500 \u00d7 \u0441\u0438\u0434\u044b [1,42,7,99]\", \"| cfg | seed | PPL |\", \"|---|---|---|\"]\n        for j in sorted([j for j in fin if j[\"status\"] == \"done\"],\n                        key=lambda j: (j[\"cfg\"][\"name\"], j[\"seed\"])):\n            c = j[\"cfg\"]\n            L.append(f\"| {j['cfg']['name']} | {j['seed']} | {j['val_ppl']} |\"\n                     f\"  (mulr={c['mulr']}, ssfull={c['ssfull']}, ssk={c['ssk']}, \"\n                     f\"ssalpha={c['ssalpha']}, lr={c['lr']}, {c.get('origin','')})\")\n        L.append(\"\")\n    L += [\"## \u041a\u043e\u043d\u0442\u0440\u043e\u043b\u044c \u043f\u0440\u0435\u0434\u0441\u043a\u0430\u0437\u0430\u043d\u0438\u0439 (E1\u2013E5 \u2014 \u0441\u043c. PREDICTIONS.md) \u2014 \u0437\u0430\u043f\u043e\u043b\u043d\u044f\u0435\u0442\u0441\u044f \u0430\u0433\u0435\u043d\u0442\u043e\u043c \u043f\u0440\u043e\u0435\u043a\u0442\u0430\"]\n    out = \"\\n\".join(L) + \"\\n\"\n    open(wp(args.workdir, \"EVO_SUMMARY.md\"), \"w\").write(out)\n    print(out)\n\n\ndef main():\n    ap = argparse.ArgumentParser()\n    sub = ap.add_subparsers(dest=\"cmd\", required=True)\n    for name in (\"init\", \"worker\", \"status\", \"summary\"):\n        sp = sub.add_parser(name)\n        sp.add_argument(\"--workdir\", default=\"evo_work\")\n        if name in (\"init\", \"worker\"):\n            sp.add_argument(\"--data\", default=\"data/prep\")\n        if name == \"worker\":\n            sp.add_argument(\"--id\", type=int, default=0)\n            sp.add_argument(\"--engine\", choices=[\"torch\", \"numpy\"], default=\"numpy\")\n            sp.add_argument(\"--backend\", default=None)\n            sp.add_argument(\"--cuda\", default=None)\n            sp.add_argument(\"--job_timeout\", type=int, default=6 * 3600)\n    args = ap.parse_args()\n    os.chdir(HERE)\n    {\"init\": cmd_init, \"worker\": cmd_worker, \"status\": cmd_status, \"summary\": cmd_summary}[args.cmd](args)\n\n\nif __name__ == \"__main__\":\n    main()\n")
print('evo.py ok')

## 2 · Данные (prep: train.npy / val.npy / meta.json)

Порядок поиска: (а) любой приаттаченный датасет `/kaggle/input/*/` с тремя файлами;
(б) локальная папка `/kaggle/working/prep_data/`; (в) скачивание с raw.githubusercontent
(нужен Internet ON). + попытка подтянуть `lc_kernels.so` для ускорения numpy-пути (необязательно).

In [ ]:
import glob, os, shutil, urllib.request

dst = os.path.join(BASE, "data", "prep")
os.makedirs(dst, exist_ok=True)
need = ("train.npy", "val.npy", "meta.json")

def have(d): return all(os.path.exists(os.path.join(d, f)) for f in need)

src = None
for d in sorted(glob.glob("/kaggle/input/*")) + ["/kaggle/working/prep_data"]:
    if have(d): src = d; break

if src:
    for f in need: shutil.copy(os.path.join(src, f), os.path.join(dst, f))
    print("данные из датасета:", src)
else:
    RAW = "https://raw.githubusercontent.com/Riyozaki/AIra/arena/01a04a42-aira/leancore/data/prep/"
    for f in need:
        print("скачиваю", f, "...")
        urllib.request.urlretrieve(RAW + f, os.path.join(dst, f))
    print("данные скачаны с GitHub raw")

# lc_kernels.so (ускоритель numpy-пути) — лучшая попытка, без падения
try:
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/Riyozaki/AIra/arena/01a04a42-aira/leancore/lc_kernels.so",
        os.path.join(BASE, "lc_kernels.so"))
    print("lc_kernels.so подтянут")
except Exception as e:
    print("lc_kernels.so не удалось (ок: numpy-фолбэк):", type(e).__name__)

import numpy as np, json as _j
tr = np.load(os.path.join(dst, "train.npy"))
print("train tokens:", len(tr), "| vocab:", _j.load(open(os.path.join(dst, 'meta.json')))['vocab'])
PREP = "data/prep"


## 3 · GPU-гейт честности и выбор движка

Если есть CUDA: прогон CTRL 40 шагов на **torch-GPU** и на **numpy-CPU** с одним сидом.
Критерий: |Δval_ppl| ≤ 2% → engine=torch (весь брекет на torch-GPU), иначе engine=numpy и
брекет едет на CPU-воркерах — но ни один бит плохих данных не попадёт в таблицу.

In [ ]:
def run_once(engine, steps=40, seed=1):
    import subprocess, json
    env = dict(os.environ); env["LC_BACKEND"] = "numpy"
    if engine == "torch":
        cmd = [sys.executable, os.path.join(BASE, "train_torch.py"), "--steps", str(steps),
               "--eval_every", "20", "--ssk", "512", "--ssfull", "0.12", "--seed", str(seed),
               "--negrng", "1", "--trunknorm", "1", "--tag", f"gate_{engine}", "--data", PREP]
    else:
        cmd = [sys.executable, os.path.join(BASE, "nano_lc_kg.py"), "--kind", "ema", "--opt", "muon",
               "--steps", str(steps), "--eval_every", "20", "--ssk", "512", "--ssfull", "0.12",
               "--seed", str(seed), "--negrng", "1", "--trunknorm", "1", "--tag", f"gate_{engine}",
               "--data", PREP]
    try:
        r = subprocess.run(cmd, cwd=BASE, env=env, capture_output=True, text=True, timeout=3600)
    except Exception as e:
        print(f"[gate:{engine}] запуск не удался: {type(e).__name__}: {e}")
        return None, []
    if r.returncode != 0:
        print(f"[gate:{engine}] КРАШ rc={r.returncode} -- хвост stderr:")
        print((r.stderr or "")[-2500:]); print("--- stdout ---"); print((r.stdout or "")[-1200:])
        return None, []
    p = os.path.join(BASE, "results", f"run_gate_{engine}.jsonl")
    if not os.path.exists(p):
        print(f"[gate:{engine}] НЕТ {p}! rc=0, но результат не записан. stderr:")
        print((r.stderr or "")[-2500:]); print("--- stdout ---"); print((r.stdout or "")[-1200:])
        return None, []
    rows = [json.loads(l) for l in open(p) if l.strip().startswith("{")]
    if not rows:
        print(f"[gate:{engine}] jsonl пуст"); return None, []
    return rows[-1]["val_ppl"], rows

NGPU, TORCH_OK = 0, False
try:
    import torch
    TORCH_OK = True
    NGPU = torch.cuda.device_count() if torch.cuda.is_available() else 0
    print("torch", torch.__version__, "| cuda устройств:", NGPU,
          [torch.cuda.get_device_name(i) for i in range(NGPU)] if NGPU else "")
except Exception as e:
    print("torch недоступен:", type(e).__name__)

ENGINE = "numpy"
WHY = f"cuda не видна из torch (NGPU={NGPU}) — если Accelerator включён, перезапусти ядро"
if NGPU > 0:
    ppl_t, rows_t = run_once("torch", steps=40, seed=1)
    if ppl_t is None:
        WHY = "torch-движок упал (лог выше) — пришли мне этот вывод"
    else:
        ppl_n, rows_n = run_once("numpy", steps=40, seed=1)
        assert ppl_n is not None, "numpy-референс не отработал (лог выше) — стоп"
        d = abs(ppl_t - ppl_n) / ppl_n
        print(f"GATE: torch={ppl_t:.2f} numpy={ppl_n:.2f} relDelta={d:.3%}")
        print(" ряды torch:", [(r["step"], r["val_ppl"]) for r in rows_t])
        print(" ряды numpy:", [(r["step"], r["val_ppl"]) for r in rows_n])
        if d <= 0.02:
            ENGINE = "torch"; WHY = f"гейт пройден (relDelta={d:.2%})"
        else:
            WHY = f"расхождение {d:.2%} > 2% — пришли мне вывод, решу вопрос"
_force = os.environ.get("LC_FORCE_ENGINE", "").strip().lower()
if _force in ("torch", "numpy") and _force != ENGINE:
    if _force == "torch" and NGPU == 0:
        print("LC_FORCE_ENGINE=torch, но cuda нет — игнорирую форс")
    else:
        print(f"ФОРС: ENGINE {ENGINE} -> {_force} (по LC_FORCE_ENGINE)")
        ENGINE = _force; WHY += " [форс]"
NWORKERS = NGPU if ENGINE == "torch" else max(1, min(3, (os.cpu_count() or 2) - 1))
print("=" * 64)
print("ENGINE:", ENGINE, "| WORKERS:", NWORKERS, "|", WHY)
if ENGINE == "numpy" and NGPU > 0:
    print("!" * 64)
    print("! GPU НЕ БУДЕТ ГРУЗИТЬСЯ, но GPU-КВОТА ГОРИТ, пока Accelerator=GPU.")
    print("! Варианты: (а) пришли мне вывод выше — чиню; или (б) Settings →")
    print("! Accelerator → None и Run All: брекет честно поедет на CPU без")
    print("! траты GPU-квоты (медленнее, но CPU-квоты хватит).")
    print("!" * 64)
print("=" * 64)


## 4 · Предсказания (зафиксированы ДО запуска — в PREDICTIONS.md)

In [ ]:
print(open(os.path.join(WORK, "PREDICTIONS.md")).read() if os.path.exists(os.path.join(WORK, "PREDICTIONS.md")) else "(запишется init'ом ниже)")

## 5 · Запуск брекета (init + воркеры + монитор)

`TARGET_HOURS` — мягкий стоп монитора (брекет сам сохраняет всё, что досчитал;
продолжение — следующая сессия). На T4×2: воркер на GPU. На P100: один. CPU-фолбэк: 3 воркера.

In [ ]:
TARGET_HOURS = 10.0   # под сессию 12ч с запасом

def shout(*a): print(*a, flush=True)

subprocess.run([sys.executable, os.path.join(BASE, "evo.py"), "init",
                "--workdir", WORK, "--data", PREP], cwd=BASE, check=True)

procs = []
for wid in range(NWORKERS):
    env = dict(os.environ)
    env["LC_BACKEND"] = "numpy"
    if ENGINE == "torch":
        cmd = [sys.executable, os.path.join(BASE, "evo.py"), "worker", "--id", str(wid),
               "--engine", "torch", "--cuda", str(wid), "--workdir", WORK, "--data", PREP]
    else:
        env["OPENBLAS_NUM_THREADS"] = str(max(1, (os.cpu_count() or 2) // max(1, NWORKERS)))
        cmd = [sys.executable, os.path.join(BASE, "evo.py"), "worker", "--id", str(wid),
               "--engine", "numpy", "--backend", "numpy", "--workdir", WORK, "--data", PREP]
    procs.append(subprocess.Popen(cmd, cwd=BASE, env=env))
    shout(f"воркер {wid} запущен (pid {procs[-1].pid})")

t0 = time.time()
while True:
    alive = sum(p.poll() is None for p in procs)
    st = subprocess.run([sys.executable, os.path.join(BASE, "evo.py"), "status",
                         "--workdir", WORK], cwd=BASE, capture_output=True, text=True).stdout.strip()
    shout(f"--- t+{(time.time()-t0)/3600:.2f}ч | воркеров живо: {alive}\n{st}")
    if alive == 0:
        shout("все воркеры завершились"); break
    if (time.time() - t0) / 3600 > TARGET_HOURS:
        shout("мягкий стоп монитора: процессы остаются, состояние сохранено"); break
    time.sleep(60)


## 6 · Итоги и артефакты

In [ ]:
subprocess.run([sys.executable, os.path.join(BASE, "evo.py"), "summary",
                "--workdir", WORK], cwd=BASE, check=True)
print("=== файлы для скачивания/продолжения ===")
for f in sorted(glob.glob(os.path.join(WORK, "*")) + glob.glob(os.path.join(BASE, "results", "*.jsonl"))
                  + glob.glob(os.path.join(BASE, "results", "ckpt_bk_*.npz"))):
    print(f"{os.path.getsize(f)/1e6:8.2f} MB  {f}")


## 7 · Продолжение после таймаута

1. **Save Version** (все файлы Output сохранятся).
2. В новой сессии этого же ноутбука: **Add Data → Notebook Output Files** предыдущей версии.
   `init` сам найдёт `evo_state.json` в `/kaggle/input/*/` и продолжит с места обрыва
   (зависшие джобы вернутся в pending, готовые не повторятся).
3. Не запускайте две сессии брекета одновременно на одном state — файловые локи не расчитаны на это.

### Что пасти в чат проекта
Содержимое `airaw/evo_work/EVO_SUMMARY.md` целиком + финальную таблицу из ячейки 6.
Я сверю с предсказаниями E1–E5, дам вердикты с метками и внесу в TRICKS.